# 卷积分类头前向计算实现与优化

卷积分类头常用于将前级网络产生的特征图映射为输出通道，并通过激活函数得到后续网络所需的特征。本实验实现单样本、NCHW布局、FP32数据类型的二维卷积、偏置累加与ReLU激活，重点研究中间结果写回Global Memory（GM）和输入数据重复搬入对性能的影响。

本实验基于Ascend C静态Tensor编程范式实现两种方案。基线方案由卷积Kernel和独立ReLU Kernel组成，卷积中间结果先写入GM，再由ReLU Kernel读回并处理；融合优化方案使用单个Kernel，在片上完成卷积后直接执行相同的ReLU，并使用环形行缓冲区复用相邻输出行所需的输入数据。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建实验目录并加载CANN环境；
3. 问题分析：分析卷积公式、数据布局、基线与融合数据流、环形行缓冲区及实验参数；
4. 核函数开发：实现基线卷积、独立ReLU和卷积ReLU融合Kernel；
5. 结果验证与性能分析：准备Host侧参考结果，完成工程构建、NPU运行、正确性验证和性能分析；
6. 实验总结：归纳算子融合、输入复用和静态LocalTensor在卷积分类头中的实现方法。


---
## 1. 实验概述

卷积和激活是卷积神经网络常见的连续计算过程，在图像分类和特征提取等任务中具有广泛应用。在并行计算中，适合用于说明数据组织、中间结果复用和算子融合之间的关系。本实验围绕卷积网络末端计算片段的前向计算展开，设置基线实现和融合优化实现两种对比实验方案。其中，基线实现按照数据展开、卷积计算、激活处理和结果写回的顺序依次完成各阶段计算；融合优化实现设计环形行缓冲区，对卷积计算所需的输入数据进行复用，并将卷积和激活过程在同一计算流程中完成，减少重复数据展开和中间结果回写。通过比较两种实验方案的计算结果、实验误差、执行时间和有效带宽，分析数据组织、中间结果复用和算子融合对计算性能的影响。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解卷积与激活的基本计算过程。掌握卷积核在输入特征图上的滑动计算方式，理解输入数据、卷积权重和输出结果之间的对应关系，并明确激活计算在卷积网络末端计算片段中的作用。
2. 掌握卷积与激活基线实现和融合优化实现的基本方法。能够理解基线实现中数据展开、卷积计算、激活处理和结果写回的顺序执行过程；掌握环形行缓冲区的组织方法，理解输入数据复用、卷积与激活融合以及中间结果写回减少的基本原理。
3. 具备正确性验证和性能分析能力。能够将Device侧计算结果与Host侧参考结果进行比较，结合最大绝对误差和最大相对误差判断计算结果是否正确；能够根据执行时间和有效带宽比较基线实现与融合优化实现的性能差异，并分析数据组织、中间结果复用和算子融合对计算性能的影响。


### 1.2 前置知识

在实验前，学生需要理解卷积和激活的基本计算过程，在此基础上进一步学习输入特征图的数据组织方式以及算子融合的基本方法。建议在实验前重点掌握如下内容。

1. 卷积计算基础：理解二维卷积的基本计算过程，掌握卷积核在输入特征图上的滑动方式，明确输入特征图、卷积权重和输出特征图之间的对应关系。
2. 激活函数基础：理解激活函数在神经网络中的作用，掌握卷积结果经过激活处理后生成输出数据的基本过程，并了解常用逐元素激活计算的特点。
3. 算子融合基础：理解将卷积和激活组织在同一计算流程中的基本思想，认识到减少中间结果写回和重复数据读取能够降低全局内存访问开销。
4. Ascend C开发基础：理解Host侧和Device侧的基本分工。Host侧负责输入数据准备、Device内存管理、Kernel启动和结果验证；Device侧负责数据搬入、片上计算和结果写回。实验前应熟悉张量操作、工程编译、脚本运行和结果查看方法。
5. 静态Tensor编程基础：理解静态Tensor编程需要确定数据块边界和LocalTensor的静态容量，并根据数据搬入、计算和写回之间的依赖关系管理同步。理解数据通常先从Global Memory搬入Local Memory，再在片上完成计算，最后写回Global Memory。


### 1.3 实验要点

实验中应重点关注以下内容：

1. 特征图数据组织：按照连续内存方式保存输入特征图、卷积权重和输出特征图，确保Host侧和Device侧对数据处理方式一致。
2. 基线实现：按照数据读取、卷积计算、激活处理和结果写回的顺序完成计算，将卷积产生的中间结果写入Global Memory，再由激活计算读取并处理。
3. 融合优化实现：将卷积和激活组织在同一计算流程中，在片上完成卷积后直接进行激活处理，减少中间结果在Global Memory中的写回和再次读取。
4. 输入数据复用：设计环形行缓冲区保存卷积计算所需的相邻输入行，在卷积核滑动过程中更新缓冲区内容，减少输入数据的重复搬入和展开。
5. 静态Tensor与同步管理：根据卷积窗口、数据块边界和片上存储容量确定LocalTensor的静态大小，并正确管理数据搬入、片上计算和结果写回之间的依赖关系。
6. 结果验证和性能分析：使用Host侧参考结果对Device侧输出进行逐项比较，分别记录基线实现和融合优化实现的执行时间、有效带宽和估算读写数据量，从数据组织、结果复用和算子融合等角度分析两种实现的性能差异。


---
## 2. 环境准备

首先创建实验所需目录，并尝试加载Ascend CANN环境变量。

- `Source/04.02/include`：公共参数、形状检查、搬运量估算和Host校验工具；
- `Source/04.02/ascend_ops/op_kernel`：基线卷积、独立ReLU和融合卷积ReLU的Device实现；
- `Source/04.02/ascend_ops/host_launch`：ACL运行时管理和Kernel启动代码；
- `Source/04.02/scripts`：构建、运行和Profiling脚本；
- `Source/04.02/results`：保存两种实验方案的运行日志与汇总结果；
- `Source/04.02/CMakeLists.txt`：配置Ascend C核函数库及NPU Host程序。

本实验不建立独立CPU仿真目标。Host程序内部的标量卷积只用于生成参考结果和执行正确性校验，不计入Device性能测量。


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess

PROJECT_ROOT = Path("Source/04.02")
for directory in [
    "include",
    "scripts",
    "results",
    "ascend_ops/op_kernel",
    "ascend_ops/host_launch",
]:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

def find_cann_root(env_script):
    for parent in [env_script.parent, *env_script.parents]:
        cmake_candidates = [
            parent / "tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
        ]
        if any(path.is_file() for path in cmake_candidates):
            return parent
    return None

candidate_scripts = []
for item in [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    os.environ.get("ASCEND_CANN_PACKAGE_PATH"),
    "/opt/conda/Ascend/cann-9.0.0",
    "/usr/local/Ascend/ascend-toolkit/latest",
]:
    if item:
        candidate_scripts.append(Path(item) / "set_env.sh")

for root in [Path("/opt/conda/Ascend"), Path("/usr/local/Ascend"),
             Path("/home/ma-user/Ascend"), Path("/opt/Ascend")]:
    if root.exists():
        candidate_scripts.extend(sorted(root.glob("**/set_env.sh"), reverse=True))

set_env = None
install_root = None
for script in candidate_scripts:
    if not script.is_file():
        continue
    root = find_cann_root(script)
    if root is not None:
        set_env = script
        install_root = root
        break

if set_env is not None and shutil.which("bash"):
    command = f"source {shlex.quote(str(set_env))} && env"
    loaded_env = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in loaded_env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(install_root)
    os.environ["ASCEND_CANN_PACKAGE_PATH"] = str(install_root)
    print("Ascend environment loaded from:", set_env)
    print("ASCEND_INSTALL_PATH:", install_root)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")

print("Experiment directory:", PROJECT_ROOT.resolve())
print("cmake:", shutil.which("cmake") or "not found")
print("C++ compiler:", shutil.which("g++") or shutil.which("c++") or "not found")


---
## 3. 问题分析

### 3.1 卷积分类头计算目标

输入特征图采用单样本NCHW布局，记为$X\in\mathbb{R}^{C_{in}\times H\times W}$；卷积权重采用OIHW布局，记为$W\in\mathbb{R}^{C_{out}\times C_{in}\times K\times K}$；偏置为$b\in\mathbb{R}^{C_{out}}$。stride固定为1，padding为$P$，输出尺寸为：

$$
H_{out}=H+2P-K+1,\qquad W_{out}=W+2P-K+1
$$

未激活卷积结果和最终输出分别为：

$$
Z_{oc,oh,ow}=b_{oc}+\sum_{ic=0}^{C_{in}-1}\sum_{kh=0}^{K-1}\sum_{kw=0}^{K-1}
X_{ic,oh+kh-P,ow+kw-P}W_{oc,ic,kh,kw}
$$

$$
Y_{oc,oh,ow}=\max(Z_{oc,oh,ow},0)
$$

输入范围外的元素按0处理。基线实验方案与融合优化实验方案使用相同的卷积累加顺序、偏置和`Maxs` ReLU，保证性能差异主要来自数据流组织。


### 3.2 数据布局、补齐与固定宽度块

输入、权重和输出均以FP32存储。GM中每条输入行、卷积核行和输出行按8个FP32元素，即32字节向上补齐。补齐区域置0，Device计算结束后Host只提取有效输出区域进行校验。

Device内部按固定8列输出宽度块处理空间维度。该宽度块是代码中的实现常量，用于组织对齐搬运、局部卷积和ReLU，不是命令行实验参数。对于每个宽度块，代码只搬入覆盖当前输出列及卷积核横向范围所需的对齐输入区间。

输出通道由`blockDim`个逻辑任务划分，每个输出通道只由一个任务写回，因此两个实验方案均不需要跨核归约或原子更新。


### 3.3 两种实验方案

| 实验方案 | Kernel数量 | 输入组织 | 卷积中间结果 | ReLU位置 | 主要GM访问 |
|---|---:|---|---|---|---|
| 基线实验方案（命令参数`basic`） | 2 | 每个输出行重新加载所需$K$行输入块 | 写入GM | 独立Kernel调用`Maxs` | 读输入/权重/偏置，写/读中间结果，写最终输出 |
| 融合优化实验方案（命令参数`fused`） | 1 | 环形行缓冲区复用相邻窗口输入行 | 保留在LocalTensor | 卷积Kernel内调用相同`Maxs` | 读输入/权重/偏置，写最终输出 |

基线方案到融合方案的变化同时体现两类优化：一是把卷积与激活组织在同一Kernel中，取消中间结果的一次GM写入和一次GM读取；二是通过环形行缓冲区复用相邻输出行共同需要的输入数据。


### 3.4 环形行缓冲区与计算依赖

卷积核高度为$K$时，相邻两个输出行的输入窗口通常共享$K-1$条输入行。基线方案在处理每个输出行时重新加载完整窗口；融合方案在LocalTensor中设置$K$个逻辑行槽位，第一次计算前装入所需行，之后窗口每向下移动一行，只覆盖最旧槽位并搬入新进入的一行。

环形槽位复用同一段UB地址，其容量由$C_{in}$、$K$和当前对齐输入块宽度决定，不随输出高度增长。`DataCopy`完成后使用`PipeBarrier<PIPE_ALL>()`保证搬运数据在卷积读取前可见；卷积完成后，`Maxs`直接处理片上输出块，再同步并写回GM。

这种依赖顺序要求当前输出行计算完成后才能覆盖仍被使用的环形槽位。环形缓冲减少的是GM重复搬入，不改变卷积数学过程。


### 3.5 公共参数与性能指标

公共头文件统一定义实验方案枚举、运行参数、形状与容量检查、CPU参考工具、误差指标、估算输入搬入次数和估算GM读写量。Device侧使用`LocalMemAllocator<AscendC::Hardware::UB>`申请`LocalTensor<float>`，Host侧在Kernel启动前保证运行参数对应的LocalTensor容量不超过静态上限。

`inputCopies`表示按照代码滑动窗口与宽度块模型估算的GM到片上存储搬入操作次数。有效带宽为：

$$
BW_{effective}=\frac{estimatedReadBytes+estimatedWriteBytes}{kernel\_us\times10^{-6}}\div10^9
$$

它用于观察当前两种数据流组织下的估算搬运速率，不等同于硬件HBM物理带宽上限或实际利用率。


In [ ]:
%%writefile Source/04.02/include/conv_head_common.h
#pragma once

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>

namespace conv_head {

constexpr uint32_t kFloatAlignment = 8;
constexpr uint32_t kFixedWidthBlock = 8;
constexpr uint32_t kMaxRingElements = 32768;
constexpr uint32_t kMaxWeightElements = 8192;
constexpr uint32_t kMaxUbFloatElements = 40000;
constexpr uint32_t kMaxPaddedRowWidth = 2048;
constexpr uint32_t kMaxPaddedOutputChannels = 4096;

enum class Version {
    Basic,
    Fused,
    All
};

struct Config {
    uint32_t input_channels = 8;
    uint32_t output_channels = 16;
    uint32_t input_height = 64;
    uint32_t input_width = 64;
    uint32_t kernel_size = 3;
    uint32_t padding = 1;
    uint32_t block_dim = 8;
    uint32_t seed = 1234;
    Version version = Version::All;
    bool sweep = false;
    bool print_output = false;
};

struct Metrics {
    double max_abs_error = 0.0;
    double max_rel_error = 0.0;
    uint64_t mismatch_count = 0;
};

struct TrafficBytes {
    double read = 0.0;
    double write = 0.0;

    double total() const {
        return read + write;
    }
};

inline const char* version_name(Version version) {
    switch (version) {
        case Version::Basic:
            return "basic";
        case Version::Fused:
            return "fused";
        case Version::All:
            return "all";
    }
    return "unknown";
}

inline Version parse_version(const std::string& value) {
    if (value == "basic") {
        return Version::Basic;
    }
    if (value == "fused") {
        return Version::Fused;
    }
    if (value == "all") {
        return Version::All;
    }
    throw std::invalid_argument("--version must be one of: basic, fused, all");
}

inline uint64_t checked_mul(uint64_t a, uint64_t b, const char* name) {
    if (a != 0 && b > std::numeric_limits<uint64_t>::max() / a) {
        throw std::overflow_error(std::string(name) + " overflows uint64_t");
    }
    return a * b;
}

inline uint64_t round_up(uint64_t value, uint64_t alignment) {
    if (alignment == 0) {
        throw std::invalid_argument("alignment must be positive");
    }
    const uint64_t remainder = value % alignment;
    if (remainder == 0) {
        return value;
    }
    const uint64_t increment = alignment - remainder;
    if (value > std::numeric_limits<uint64_t>::max() - increment) {
        throw std::overflow_error("round_up overflows uint64_t");
    }
    return value + increment;
}

inline size_t float_bytes(uint64_t count) {
    if (count > static_cast<uint64_t>(std::numeric_limits<size_t>::max() /
                                      sizeof(float))) {
        throw std::overflow_error("float allocation size overflows size_t");
    }
    return static_cast<size_t>(count) * sizeof(float);
}

inline uint32_t output_extent(uint32_t input_extent,
                              uint32_t kernel_size,
                              uint32_t padding) {
    const uint64_t padded =
        static_cast<uint64_t>(input_extent) + 2ull * padding;
    if (padded < kernel_size) {
        throw std::invalid_argument(
            "kernel_size is larger than the padded input extent");
    }
    const uint64_t extent = padded - kernel_size + 1ull;
    if (extent > std::numeric_limits<uint32_t>::max()) {
        throw std::overflow_error("output extent exceeds uint32_t");
    }
    return static_cast<uint32_t>(extent);
}

inline uint32_t padded_width(uint32_t width) {
    const uint64_t padded = round_up(width, kFloatAlignment);
    if (padded > std::numeric_limits<uint32_t>::max()) {
        throw std::overflow_error("padded width exceeds uint32_t");
    }
    return static_cast<uint32_t>(padded);
}

inline uint64_t valid_input_elements(const Config& cfg) {
    return checked_mul(
        checked_mul(cfg.input_channels, cfg.input_height, "input elements"),
        cfg.input_width,
        "input elements");
}

inline uint64_t valid_weight_elements(const Config& cfg) {
    return checked_mul(
        checked_mul(cfg.output_channels, cfg.input_channels, "weight elements"),
        checked_mul(cfg.kernel_size, cfg.kernel_size, "kernel elements"),
        "weight elements");
}

inline uint64_t valid_output_elements(const Config& cfg) {
    const uint32_t output_height =
        output_extent(cfg.input_height, cfg.kernel_size, cfg.padding);
    const uint32_t output_width =
        output_extent(cfg.input_width, cfg.kernel_size, cfg.padding);
    return checked_mul(
        checked_mul(cfg.output_channels, output_height, "output elements"),
        output_width,
        "output elements");
}

inline uint32_t active_core_count(uint32_t output_channels,
                                  uint32_t block_dim) {
    return std::min(output_channels, block_dim);
}

inline uint32_t aligned_block_input_span(
    const Config& cfg,
    uint32_t output_column_begin,
    uint32_t current_width_block) {
    const int64_t required_begin =
        static_cast<int64_t>(output_column_begin) -
        static_cast<int64_t>(cfg.padding);
    const int64_t required_end =
        static_cast<int64_t>(output_column_begin) +
        static_cast<int64_t>(current_width_block) +
        static_cast<int64_t>(cfg.kernel_size) -
        static_cast<int64_t>(cfg.padding) - 1;
    const int64_t valid_begin =
        std::max<int64_t>(0, required_begin);
    const int64_t valid_end =
        std::min<int64_t>(cfg.input_width, required_end);
    if (valid_begin >= valid_end) {
        return 0;
    }

    const uint64_t aligned_begin =
        static_cast<uint64_t>(valid_begin) /
        kFloatAlignment * kFloatAlignment;
    const uint64_t aligned_end =
        round_up(static_cast<uint64_t>(valid_end),
                 kFloatAlignment);
    return static_cast<uint32_t>(aligned_end - aligned_begin);
}

inline uint32_t fixed_ring_row_width(const Config& cfg,
                                     uint32_t output_width) {
    uint32_t max_span = 0;
    for (uint32_t output_column_begin = 0;
         output_column_begin < output_width;
         output_column_begin += kFixedWidthBlock) {
        const uint32_t current_width_block =
            std::min(kFixedWidthBlock,
                     output_width - output_column_begin);
        max_span = std::max(
            max_span,
            aligned_block_input_span(cfg,
                                     output_column_begin,
                                     current_width_block));
    }
    return max_span;
}

inline uint64_t fixed_block_input_elements_per_channel_row(
    const Config& cfg,
    uint32_t output_width) {
    uint64_t elements = 0;
    for (uint32_t output_column_begin = 0;
         output_column_begin < output_width;
         output_column_begin += kFixedWidthBlock) {
        const uint32_t current_width_block =
            std::min(kFixedWidthBlock,
                     output_width - output_column_begin);
        elements += aligned_block_input_span(
            cfg,
            output_column_begin,
            current_width_block);
    }
    return elements;
}

inline uint64_t nonempty_fixed_block_count(
    const Config& cfg,
    uint32_t output_width) {
    uint64_t count = 0;
    for (uint32_t output_column_begin = 0;
         output_column_begin < output_width;
         output_column_begin += kFixedWidthBlock) {
        const uint32_t current_width_block =
            std::min(kFixedWidthBlock,
                     output_width - output_column_begin);
        if (aligned_block_input_span(cfg,
                                     output_column_begin,
                                     current_width_block) != 0) {
            ++count;
        }
    }
    return count;
}

inline uint64_t baseline_valid_window_row_count(
    const Config& cfg,
    uint32_t output_height) {
    uint64_t valid_rows = 0;
    for (uint32_t output_row = 0;
         output_row < output_height;
         ++output_row) {
        for (uint32_t kernel_row = 0;
             kernel_row < cfg.kernel_size;
             ++kernel_row) {
            const int64_t input_row =
                static_cast<int64_t>(output_row) +
                static_cast<int64_t>(kernel_row) -
                static_cast<int64_t>(cfg.padding);
            if (input_row >= 0 &&
                input_row <
                    static_cast<int64_t>(cfg.input_height)) {
                ++valid_rows;
            }
        }
    }
    return valid_rows;
}

inline uint64_t estimated_input_copy_count(const Config& cfg,
                                           Version version) {
    const uint32_t output_height =
        output_extent(cfg.input_height,
                      cfg.kernel_size,
                      cfg.padding);
    const uint32_t output_width =
        output_extent(cfg.input_width,
                      cfg.kernel_size,
                      cfg.padding);
    if (version == Version::Basic) {
        return checked_mul(
            checked_mul(
                checked_mul(
                    baseline_valid_window_row_count(
                        cfg,
                        output_height),
                    nonempty_fixed_block_count(
                        cfg,
                        output_width),
                    "baseline input copy count"),
                cfg.output_channels,
                "baseline input copy count"),
            cfg.input_channels,
            "baseline input copy count");
    }
    if (version == Version::Fused) {
        return checked_mul(
            checked_mul(
                checked_mul(
                    nonempty_fixed_block_count(
                        cfg,
                        output_width),
                    cfg.input_height,
                    "fused input copy count"),
                cfg.input_channels,
                "fused input copy count"),
            cfg.output_channels,
            "fused input copy count");
    }
    throw std::invalid_argument(
        "estimated_input_copy_count requires basic or fused");
}

inline void check_config(const Config& cfg) {
    if (cfg.input_channels == 0 || cfg.output_channels == 0 ||
        cfg.input_height == 0 || cfg.input_width == 0 ||
        cfg.kernel_size == 0) {
        throw std::invalid_argument(
            "channels, spatial extents, and kernel_size must be positive");
    }
    if (cfg.block_dim == 0) {
        throw std::invalid_argument("--block-dim must be positive");
    }

    const uint32_t output_height =
        output_extent(cfg.input_height, cfg.kernel_size, cfg.padding);
    const uint32_t output_width =
        output_extent(cfg.input_width, cfg.kernel_size, cfg.padding);
    const uint32_t padded_input_width = padded_width(cfg.input_width);
    const uint32_t padded_output_width = padded_width(output_width);
    const uint32_t padded_kernel_width = padded_width(cfg.kernel_size);
    const uint32_t padded_output_channels =
        padded_width(cfg.output_channels);
    if (output_height == 0 || output_width == 0) {
        throw std::invalid_argument("output spatial extent must be positive");
    }
    if (padded_input_width > kMaxPaddedRowWidth ||
        padded_output_width > kMaxPaddedRowWidth) {
        throw std::invalid_argument(
            "padded input/output row width exceeds the static limit 2048");
    }
    if (padded_output_channels > kMaxPaddedOutputChannels) {
        throw std::invalid_argument(
            "padded output channels exceed the static limit 4096");
    }

    const uint32_t ring_row_width =
        fixed_ring_row_width(cfg, output_width);
    const uint64_t ring_elements = checked_mul(
        checked_mul(cfg.input_channels,
                    cfg.kernel_size,
                    "ring-buffer elements"),
        ring_row_width,
        "ring-buffer elements");
    const uint64_t weight_per_output_channel = checked_mul(
        checked_mul(cfg.input_channels,
                    cfg.kernel_size,
                    "weight elements per output channel"),
        padded_kernel_width,
        "weight elements per output channel");
    if (ring_elements > kMaxRingElements) {
        throw std::invalid_argument(
            "the fixed 8-column block exceeds the ring-buffer limit 32768");
    }
    if (weight_per_output_channel > kMaxWeightElements) {
        throw std::invalid_argument(
            "Cin * K * padded_kernel_width exceeds 8192");
    }

    const uint64_t convolution_ub_elements =
        ring_elements + weight_per_output_channel +
        padded_output_channels + kFixedWidthBlock;
    if (convolution_ub_elements > kMaxUbFloatElements) {
        throw std::invalid_argument(
            "requested convolution LocalTensor storage exceeds 40000 FP32 elements");
    }

    const uint64_t padded_input_count = checked_mul(
        checked_mul(cfg.input_channels,
                    cfg.input_height,
                    "padded input elements"),
        padded_input_width,
        "padded input elements");
    const uint64_t padded_weight_count = checked_mul(
        cfg.output_channels,
        weight_per_output_channel,
        "padded weight elements");
    const uint64_t padded_output_count = checked_mul(
        checked_mul(cfg.output_channels,
                    output_height,
                    "padded output elements"),
        padded_output_width,
        "padded output elements");
    const uint64_t max_kernel_count = std::max(
        padded_input_count,
        std::max(padded_weight_count, padded_output_count));
    if (max_kernel_count > std::numeric_limits<uint32_t>::max()) {
        throw std::invalid_argument(
            "a padded tensor exceeds the uint32_t kernel address range");
    }
}

inline TrafficBytes estimated_traffic(const Config& cfg,
                                      Version version) {
    const uint32_t output_height =
        output_extent(cfg.input_height, cfg.kernel_size, cfg.padding);
    const uint32_t output_width =
        output_extent(cfg.input_width, cfg.kernel_size, cfg.padding);
    const uint32_t padded_output_width = padded_width(output_width);
    const uint32_t padded_kernel_width = padded_width(cfg.kernel_size);
    const uint32_t padded_output_channels =
        padded_width(cfg.output_channels);

    const uint64_t weight_elements = checked_mul(
        checked_mul(cfg.output_channels,
                    cfg.input_channels,
                    "estimated weight elements"),
        checked_mul(cfg.kernel_size,
                    padded_kernel_width,
                    "estimated weight elements"),
        "estimated weight elements");
    const uint64_t padded_output_elements = checked_mul(
        checked_mul(cfg.output_channels,
                    output_height,
                    "estimated output elements"),
        padded_output_width,
        "estimated output elements");
    const uint64_t bias_reads = checked_mul(
        active_core_count(cfg.output_channels, cfg.block_dim),
        padded_output_channels,
        "estimated bias reads");

    uint64_t read_float_elements = 0;
    uint64_t write_float_elements = 0;
    if (version == Version::Basic) {
        const uint64_t window_reads = checked_mul(
            checked_mul(
                baseline_valid_window_row_count(cfg,
                                                output_height),
                cfg.output_channels,
                "baseline input reads"),
            checked_mul(
                cfg.input_channels,
                fixed_block_input_elements_per_channel_row(
                    cfg,
                    output_width),
                "baseline input reads"),
            "baseline input reads");
        read_float_elements =
            window_reads + weight_elements + bias_reads +
            padded_output_elements;
        write_float_elements =
            2ull * padded_output_elements;
    } else if (version == Version::Fused) {
        const uint64_t ring_reads = checked_mul(
            checked_mul(
                checked_mul(cfg.output_channels,
                            cfg.input_height,
                            "fused input reads"),
                cfg.input_channels,
                "fused input reads"),
            fixed_block_input_elements_per_channel_row(
                cfg,
                output_width),
            "fused input reads");
        read_float_elements =
            ring_reads + weight_elements + bias_reads;
        write_float_elements = padded_output_elements;
    } else {
        throw std::invalid_argument(
            "estimated_traffic requires basic or fused");
    }
    TrafficBytes traffic;
    traffic.read =
        static_cast<double>(read_float_elements) * sizeof(float);
    traffic.write =
        static_cast<double>(write_float_elements) * sizeof(float);
    return traffic;
}

inline double estimated_moved_bytes(const Config& cfg, Version version) {
    return estimated_traffic(cfg, version).total();
}

inline double effective_gbps(const Config& cfg,
                             Version version,
                             double kernel_us) {
    if (kernel_us <= 0.0) {
        return 0.0;
    }
    return estimated_moved_bytes(cfg, version) /
           (kernel_us * 1.0e-6) / 1.0e9;
}

inline void print_header() {
    std::cout << std::setw(8) << "Cin"
              << std::setw(8) << "Cout"
              << std::setw(8) << "H"
              << std::setw(8) << "W"
              << std::setw(6) << "K"
              << std::setw(8) << "pad"
              << std::setw(14) << "inputCopies"
              << std::setw(10) << "version"
              << std::setw(14) << "kernel_us"
              << std::setw(14) << "read_MiB"
              << std::setw(14) << "write_MiB"
              << std::setw(12) << "GB/s"
              << std::setw(14) << "max_abs"
              << std::setw(14) << "max_rel"
              << std::setw(10) << "errors"
              << std::setw(10) << "status"
              << "\n";
}

inline void print_result(const Config& cfg,
                         Version version,
                         double kernel_us,
                         const Metrics& metrics) {
    const TrafficBytes traffic = estimated_traffic(cfg, version);
    std::cout << std::setw(8) << cfg.input_channels
              << std::setw(8) << cfg.output_channels
              << std::setw(8) << cfg.input_height
              << std::setw(8) << cfg.input_width
              << std::setw(6) << cfg.kernel_size
              << std::setw(8) << cfg.padding;
    std::cout << std::setw(14)
              << estimated_input_copy_count(cfg, version)
              << std::setw(10) << version_name(version)
              << std::setw(14) << std::fixed << std::setprecision(2)
              << kernel_us
              << std::setw(14) << std::fixed << std::setprecision(3)
              << traffic.read / (1024.0 * 1024.0)
              << std::setw(14) << std::fixed << std::setprecision(3)
              << traffic.write / (1024.0 * 1024.0)
              << std::setw(12) << std::fixed << std::setprecision(3)
              << effective_gbps(cfg, version, kernel_us)
              << std::setw(14) << std::scientific << std::setprecision(3)
              << metrics.max_abs_error
              << std::setw(14) << std::scientific << std::setprecision(3)
              << metrics.max_rel_error
              << std::setw(10) << std::defaultfloat
              << metrics.mismatch_count
              << std::setw(10)
              << (metrics.mismatch_count == 0 ? "PASS" : "FAIL")
              << "\n";
}

}  // namespace conv_head


---
## 4. 核函数开发

### 4.1 公共分块、搬运与标量卷积函数

三个Device入口共用相同的输出通道划分、输入块范围计算、输入行搬运和标量卷积函数。`GetContiguousRange`把连续输出通道分配给不同逻辑任务；`GetAlignedInputBlockRange`计算当前输出宽度块所需的对齐输入范围；`LoadInputRowBlock`完成GM到LocalTensor的数据搬入。

`ComputeConvolutionBlock`不是Ascend C内置接口，而是本实验封装的共享Device辅助函数。它按照输出列、输入通道、卷积核行和卷积核列执行相同顺序的FP32标量乘加，使基线方案与融合方案的卷积主体保持一致。


In [ ]:
%%writefile Source/04.02/ascend_ops/op_kernel/conv_head_static_tensor.cpp
#include "kernel_operator.h"

using namespace AscendC;

namespace {

constexpr uint32_t kFloatAlignment = 8;
constexpr uint32_t kFixedWidthBlock = 8;
constexpr uint32_t kMaxRingElements = 32768;
constexpr uint32_t kMaxWeightElements = 8192;
constexpr uint32_t kMaxUbFloatElements = 40000;
constexpr uint32_t kMaxPaddedRowWidth = 2048;
constexpr uint32_t kMaxPaddedOutputChannels = 4096;

__aicore__ inline uint32_t MinU32(uint32_t lhs, uint32_t rhs) {
    return lhs < rhs ? lhs : rhs;
}

__aicore__ inline uint32_t RoundUpU32(uint32_t value,
                                      uint32_t alignment) {
    return (value + alignment - 1) / alignment * alignment;
}

__aicore__ inline void GetContiguousRange(uint32_t count,
                                          uint32_t launchBlockDim,
                                          uint32_t& begin,
                                          uint32_t& end) {
    const uint32_t coreCount =
        launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreCount) {
        begin = 0;
        end = 0;
        return;
    }
    const uint32_t baseItems = count / coreCount;
    const uint32_t remainderItems = count % coreCount;
    begin = coreId * baseItems +
            MinU32(coreId, remainderItems);
    end = begin + baseItems +
          (coreId < remainderItems ? 1u : 0u);
}

__aicore__ inline bool CommonArgumentsAreValid(
    uint32_t inputChannels,
    uint32_t outputChannels,
    uint32_t inputHeight,
    uint32_t inputWidth,
    uint32_t outputHeight,
    uint32_t outputWidth,
    uint32_t kernelSize,
    uint32_t paddedInputWidth,
    uint32_t paddedOutputWidth,
    uint32_t paddedKernelWidth,
    uint32_t paddedOutputChannels) {
    if (inputChannels == 0 || outputChannels == 0 ||
        inputHeight == 0 || inputWidth == 0 ||
        outputHeight == 0 || outputWidth == 0 ||
        kernelSize == 0) {
        return false;
    }
    if (paddedInputWidth < inputWidth ||
        paddedOutputWidth < outputWidth ||
        paddedKernelWidth < kernelSize ||
        paddedOutputChannels < outputChannels) {
        return false;
    }
    if ((paddedInputWidth % kFloatAlignment) != 0 ||
        (paddedOutputWidth % kFloatAlignment) != 0 ||
        (paddedKernelWidth % kFloatAlignment) != 0 ||
        (paddedOutputChannels % kFloatAlignment) != 0) {
        return false;
    }
    if (paddedInputWidth > kMaxPaddedRowWidth ||
        paddedOutputWidth > kMaxPaddedRowWidth ||
        paddedOutputChannels > kMaxPaddedOutputChannels) {
        return false;
    }
    return true;
}

__aicore__ inline void GetAlignedInputBlockRange(
    uint32_t outputColumnBegin,
    uint32_t currentWidthBlock,
    uint32_t inputWidth,
    uint32_t kernelSize,
    uint32_t padding,
    uint32_t& alignedInputBegin,
    uint32_t& copyElements) {
    const int32_t requiredBegin =
        static_cast<int32_t>(outputColumnBegin) -
        static_cast<int32_t>(padding);
    const int32_t requiredEnd =
        static_cast<int32_t>(outputColumnBegin) +
        static_cast<int32_t>(currentWidthBlock) +
        static_cast<int32_t>(kernelSize) -
        static_cast<int32_t>(padding) - 1;
    const int32_t validBegin =
        requiredBegin > 0 ? requiredBegin : 0;
    const int32_t validEnd =
        requiredEnd < static_cast<int32_t>(inputWidth)
            ? requiredEnd
            : static_cast<int32_t>(inputWidth);
    if (validBegin >= validEnd) {
        alignedInputBegin = 0;
        copyElements = 0;
        return;
    }

    alignedInputBegin =
        static_cast<uint32_t>(validBegin) /
        kFloatAlignment * kFloatAlignment;
    const uint32_t alignedInputEnd =
        RoundUpU32(static_cast<uint32_t>(validEnd),
                   kFloatAlignment);
    copyElements = alignedInputEnd - alignedInputBegin;
}

__aicore__ inline void LoadInputRowBlock(
    LocalTensor<float> ringLocal,
    GlobalTensor<float> inputGm,
    int32_t inputRow,
    uint32_t slot,
    uint32_t alignedInputBegin,
    uint32_t copyElements,
    uint32_t inputChannels,
    uint32_t inputHeight,
    uint32_t paddedInputWidth,
    uint32_t paddedRingWidth) {
    const uint32_t channelBlockElements =
        inputChannels * paddedRingWidth;
    const uint32_t localBase =
        slot * channelBlockElements;
    if (inputRow < 0 ||
        inputRow >= static_cast<int32_t>(inputHeight)) {
        Duplicate(ringLocal[localBase],
                  0.0f,
                  channelBlockElements);
        PipeBarrier<PIPE_ALL>();
        return;
    }
    if (copyElements == 0) {
        PipeBarrier<PIPE_ALL>();
        return;
    }

    for (uint32_t inputChannel = 0;
         inputChannel < inputChannels;
         ++inputChannel) {
        const uint32_t globalBase =
            (inputChannel * inputHeight +
             static_cast<uint32_t>(inputRow)) *
                paddedInputWidth +
            alignedInputBegin;
        const uint32_t localChannelBase =
            localBase + inputChannel * paddedRingWidth;
        DataCopy(ringLocal[localChannelBase],
                 inputGm[globalBase],
                 copyElements);
    }
    PipeBarrier<PIPE_ALL>();
}

__aicore__ inline void ComputeConvolutionBlock(
    LocalTensor<float> outputBlockLocal,
    LocalTensor<float> ringLocal,
    LocalTensor<float> weightLocal,
    float bias,
    uint32_t ringStart,
    uint32_t alignedInputBegin,
    uint32_t outputColumnBegin,
    uint32_t currentWidthBlock,
    uint32_t inputChannels,
    uint32_t inputWidth,
    uint32_t paddedRingWidth,
    uint32_t paddedOutputBlockWidth,
    uint32_t kernelSize,
    uint32_t paddedKernelWidth,
    uint32_t padding) {
    const uint32_t channelBlockElements =
        inputChannels * paddedRingWidth;
    if (currentWidthBlock < paddedOutputBlockWidth) {
        Duplicate(outputBlockLocal,
                  0.0f,
                  paddedOutputBlockWidth);
        PipeBarrier<PIPE_ALL>();
    }

    if (kernelSize == 3) {
        // K=3 is the experiment's normal configuration. Resolve the
        // three physical ring slots once per output row instead of
        // evaluating a modulo in the innermost MAC loops.
        const uint32_t ringSlot0 = ringStart;
        uint32_t ringSlot1 = ringSlot0 + 1;
        if (ringSlot1 >= 3) {
            ringSlot1 -= 3;
        }
        uint32_t ringSlot2 = ringSlot1 + 1;
        if (ringSlot2 >= 3) {
            ringSlot2 -= 3;
        }
        const uint32_t ringBase0 =
            ringSlot0 * channelBlockElements;
        const uint32_t ringBase1 =
            ringSlot1 * channelBlockElements;
        const uint32_t ringBase2 =
            ringSlot2 * channelBlockElements;

        for (uint32_t blockColumn = 0;
             blockColumn < currentWidthBlock;
             ++blockColumn) {
            const uint32_t outputColumn =
                outputColumnBegin + blockColumn;
            float accumulator = bias;
            for (uint32_t inputChannel = 0;
                 inputChannel < inputChannels;
                ++inputChannel) {
                const uint32_t channelRingBase =
                    inputChannel * paddedRingWidth;
                const uint32_t inputRowBase0 =
                    ringBase0 + channelRingBase;
                const uint32_t inputRowBase1 =
                    ringBase1 + channelRingBase;
                const uint32_t inputRowBase2 =
                    ringBase2 + channelRingBase;
                const uint32_t channelWeightBase =
                    inputChannel * 3 * paddedKernelWidth;
                const uint32_t weightRowBase0 =
                    channelWeightBase;
                const uint32_t weightRowBase1 =
                    channelWeightBase + paddedKernelWidth;
                const uint32_t weightRowBase2 =
                    channelWeightBase +
                    2 * paddedKernelWidth;

                for (uint32_t kernelColumn = 0;
                     kernelColumn < 3;
                     ++kernelColumn) {
                    const int32_t inputColumn =
                        static_cast<int32_t>(outputColumn) +
                        static_cast<int32_t>(kernelColumn) -
                        static_cast<int32_t>(padding);
                    if (inputColumn >= 0 &&
                        inputColumn <
                            static_cast<int32_t>(inputWidth)) {
                        const uint32_t localInputColumn =
                            static_cast<uint32_t>(inputColumn) -
                            alignedInputBegin;
                        accumulator +=
                            ringLocal.GetValue(
                                inputRowBase0 +
                                localInputColumn) *
                            weightLocal.GetValue(
                                weightRowBase0 +
                                kernelColumn);
                        accumulator +=
                            ringLocal.GetValue(
                                inputRowBase1 +
                                localInputColumn) *
                            weightLocal.GetValue(
                                weightRowBase1 +
                                kernelColumn);
                        accumulator +=
                            ringLocal.GetValue(
                                inputRowBase2 +
                                localInputColumn) *
                            weightLocal.GetValue(
                                weightRowBase2 +
                                kernelColumn);
                    }
                }
            }
            outputBlockLocal.SetValue(blockColumn,
                                      accumulator);
        }
    } else {
        for (uint32_t blockColumn = 0;
             blockColumn < currentWidthBlock;
             ++blockColumn) {
            const uint32_t outputColumn =
                outputColumnBegin + blockColumn;
            float accumulator = bias;
            for (uint32_t inputChannel = 0;
                 inputChannel < inputChannels;
                 ++inputChannel) {
                uint32_t logicalKernelRow = 0;

                // Visit the cyclic ring as two contiguous ranges.
                // This avoids division/modulo for arbitrary K.
                for (uint32_t ringSlot = ringStart;
                     ringSlot < kernelSize;
                     ++ringSlot, ++logicalKernelRow) {
                    const uint32_t inputRowBase =
                        ringSlot * channelBlockElements +
                        inputChannel * paddedRingWidth;
                    const uint32_t weightRowBase =
                        (inputChannel * kernelSize +
                         logicalKernelRow) *
                        paddedKernelWidth;
                    for (uint32_t kernelColumn = 0;
                         kernelColumn < kernelSize;
                         ++kernelColumn) {
                        const int32_t inputColumn =
                            static_cast<int32_t>(outputColumn) +
                            static_cast<int32_t>(kernelColumn) -
                            static_cast<int32_t>(padding);
                        if (inputColumn >= 0 &&
                            inputColumn <
                                static_cast<int32_t>(inputWidth)) {
                            const uint32_t localInputColumn =
                                static_cast<uint32_t>(inputColumn) -
                                alignedInputBegin;
                            accumulator +=
                                ringLocal.GetValue(
                                    inputRowBase +
                                    localInputColumn) *
                                weightLocal.GetValue(
                                    weightRowBase +
                                    kernelColumn);
                        }
                    }
                }
                for (uint32_t ringSlot = 0;
                     ringSlot < ringStart;
                     ++ringSlot, ++logicalKernelRow) {
                    const uint32_t inputRowBase =
                        ringSlot * channelBlockElements +
                        inputChannel * paddedRingWidth;
                    const uint32_t weightRowBase =
                        (inputChannel * kernelSize +
                         logicalKernelRow) *
                        paddedKernelWidth;
                    for (uint32_t kernelColumn = 0;
                         kernelColumn < kernelSize;
                         ++kernelColumn) {
                        const int32_t inputColumn =
                            static_cast<int32_t>(outputColumn) +
                            static_cast<int32_t>(kernelColumn) -
                            static_cast<int32_t>(padding);
                        if (inputColumn >= 0 &&
                            inputColumn <
                                static_cast<int32_t>(inputWidth)) {
                            const uint32_t localInputColumn =
                                static_cast<uint32_t>(inputColumn) -
                                alignedInputBegin;
                            accumulator +=
                                ringLocal.GetValue(
                                    inputRowBase +
                                    localInputColumn) *
                                weightLocal.GetValue(
                                    weightRowBase +
                                    kernelColumn);
                        }
                    }
                }
            }
            outputBlockLocal.SetValue(blockColumn,
                                      accumulator);
        }
    }
    PipeBarrier<PIPE_ALL>();
}

}  // namespace



### 4.2 基线实验方案：卷积Kernel写回中间结果

`conv_head_baseline_conv`在UB中申请输入窗口、权重、偏置和输出块LocalTensor。它对每个输出行重新搬入$K$条有效输入行块，调用共享标量卷积函数完成卷积与偏置累加，再将未激活结果写入GM中的`intermediate`缓冲区。

该Kernel故意不执行ReLU，用于保留“卷积中间结果写回GM”的基线流程。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/op_kernel/conv_head_static_tensor.cpp
extern "C" __global__ __aicore__ void conv_head_baseline_conv(
    GM_ADDR input,
    GM_ADDR weight,
    GM_ADDR bias,
    GM_ADDR intermediate,
    uint32_t inputChannels,
    uint32_t outputChannels,
    uint32_t inputHeight,
    uint32_t inputWidth,
    uint32_t outputHeight,
    uint32_t outputWidth,
    uint32_t kernelSize,
    uint32_t padding,
    uint32_t paddedInputWidth,
    uint32_t paddedOutputWidth,
    uint32_t paddedKernelWidth,
    uint32_t paddedOutputChannels,
    uint32_t paddedRingWidth,
    uint32_t launchBlockDim) {
    InitSocState();

    if (!CommonArgumentsAreValid(inputChannels,
                                 outputChannels,
                                 inputHeight,
                                 inputWidth,
                                 outputHeight,
                                 outputWidth,
                                 kernelSize,
                                 paddedInputWidth,
                                 paddedOutputWidth,
                                 paddedKernelWidth,
                                 paddedOutputChannels) ||
        paddedRingWidth == 0 ||
        (paddedRingWidth % kFloatAlignment) != 0) {
        return;
    }

    const uint32_t paddedOutputBlockWidth =
        kFixedWidthBlock;
    const uint32_t windowElements =
        inputChannels * kernelSize * paddedRingWidth;
    const uint32_t weightPerOutputChannel =
        inputChannels * kernelSize * paddedKernelWidth;
    const uint32_t ubElements =
        windowElements + weightPerOutputChannel +
        paddedOutputChannels + paddedOutputBlockWidth;
    if (windowElements > kMaxRingElements ||
        weightPerOutputChannel > kMaxWeightElements ||
        ubElements > kMaxUbFloatElements) {
        return;
    }

    const uint32_t inputElements =
        inputChannels * inputHeight * paddedInputWidth;
    const uint32_t weightElements =
        outputChannels * weightPerOutputChannel;
    const uint32_t outputElements =
        outputChannels * outputHeight * paddedOutputWidth;

    GlobalTensor<float> inputGm;
    GlobalTensor<float> weightGm;
    GlobalTensor<float> biasGm;
    GlobalTensor<float> intermediateGm;
    inputGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(input),
        inputElements);
    weightGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(weight),
        weightElements);
    biasGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(bias),
        paddedOutputChannels);
    intermediateGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(intermediate),
        outputElements);

    uint32_t beginOutputChannel = 0;
    uint32_t endOutputChannel = 0;
    GetContiguousRange(outputChannels,
                       launchBlockDim,
                       beginOutputChannel,
                       endOutputChannel);
    if (beginOutputChannel >= endOutputChannel) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> windowLocal =
        ubAllocator.Alloc<float>(windowElements);
    LocalTensor<float> weightLocal =
        ubAllocator.Alloc<float>(weightPerOutputChannel);
    LocalTensor<float> biasLocal =
        ubAllocator.Alloc<float>(paddedOutputChannels);
    LocalTensor<float> outputBlockLocal =
        ubAllocator.Alloc<float>(paddedOutputBlockWidth);
    windowLocal.SetSize(windowElements);
    weightLocal.SetSize(weightPerOutputChannel);
    biasLocal.SetSize(paddedOutputChannels);
    outputBlockLocal.SetSize(paddedOutputBlockWidth);

    DataCopy(biasLocal, biasGm, paddedOutputChannels);
    PipeBarrier<PIPE_ALL>();

    for (uint32_t outputChannel = beginOutputChannel;
         outputChannel < endOutputChannel;
         ++outputChannel) {
        DataCopy(weightLocal,
                 weightGm[
                     outputChannel * weightPerOutputChannel],
                 weightPerOutputChannel);
        PipeBarrier<PIPE_ALL>();
        const float channelBias =
            biasLocal.GetValue(outputChannel);

        for (uint32_t outputColumnBegin = 0;
             outputColumnBegin < outputWidth;
             outputColumnBegin += kFixedWidthBlock) {
            const uint32_t currentWidthBlock =
                MinU32(kFixedWidthBlock,
                       outputWidth - outputColumnBegin);
            const uint32_t currentPaddedOutputBlockWidth =
                RoundUpU32(currentWidthBlock,
                           kFloatAlignment);
            uint32_t alignedInputBegin = 0;
            uint32_t copyElements = 0;
            GetAlignedInputBlockRange(outputColumnBegin,
                                      currentWidthBlock,
                                      inputWidth,
                                      kernelSize,
                                      padding,
                                      alignedInputBegin,
                                      copyElements);
            if (copyElements > paddedRingWidth) {
                return;
            }

            for (uint32_t outputRow = 0;
                 outputRow < outputHeight;
                 ++outputRow) {
                // Baseline behavior: every output row reloads all K
                // row blocks. Adjacent rows therefore reload the K-1
                // blocks that their convolution windows share.
                for (uint32_t kernelRow = 0;
                     kernelRow < kernelSize;
                     ++kernelRow) {
                    const int32_t inputRow =
                        static_cast<int32_t>(outputRow) +
                        static_cast<int32_t>(kernelRow) -
                        static_cast<int32_t>(padding);
                    LoadInputRowBlock(windowLocal,
                                      inputGm,
                                      inputRow,
                                      kernelRow,
                                      alignedInputBegin,
                                      copyElements,
                                      inputChannels,
                                      inputHeight,
                                      paddedInputWidth,
                                      paddedRingWidth);
                }

                ComputeConvolutionBlock(
                    outputBlockLocal,
                    windowLocal,
                    weightLocal,
                    channelBias,
                    0,
                    alignedInputBegin,
                    outputColumnBegin,
                    currentWidthBlock,
                    inputChannels,
                    inputWidth,
                    paddedRingWidth,
                    currentPaddedOutputBlockWidth,
                    kernelSize,
                    paddedKernelWidth,
                    padding);
                const uint32_t outputBase =
                    (outputChannel * outputHeight +
                     outputRow) *
                        paddedOutputWidth +
                    outputColumnBegin;
                DataCopy(intermediateGm[outputBase],
                         outputBlockLocal,
                         currentPaddedOutputBlockWidth);
                PipeBarrier<PIPE_ALL>();
            }
        }
    }
}



### 4.3 基线实验方案：独立ReLU Kernel

`conv_head_baseline_relu`从GM读取卷积中间结果，按固定宽度块搬入`outputBlockLocal`，调用`Maxs(outputBlockLocal, outputBlockLocal, 0.0f, len)`实现ReLU，再将最终结果写回GM。

该Kernel与基线卷积Kernel连续启动，两者执行时间及流同步共同计入`basic`方案的`kernel_us`。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/op_kernel/conv_head_static_tensor.cpp
extern "C" __global__ __aicore__ void conv_head_baseline_relu(
    GM_ADDR intermediate,
    GM_ADDR output,
    uint32_t outputChannels,
    uint32_t outputHeight,
    uint32_t outputWidth,
    uint32_t paddedOutputWidth,
    uint32_t launchBlockDim) {
    InitSocState();

    if (outputChannels == 0 || outputHeight == 0 ||
        outputWidth == 0 || paddedOutputWidth < outputWidth ||
        (paddedOutputWidth % kFloatAlignment) != 0 ||
        paddedOutputWidth > kMaxPaddedRowWidth) {
        return;
    }

    const uint32_t rowCount =
        outputChannels * outputHeight;
    const uint32_t outputElements =
        rowCount * paddedOutputWidth;
    GlobalTensor<float> intermediateGm;
    GlobalTensor<float> outputGm;
    intermediateGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(intermediate),
        outputElements);
    outputGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(output),
        outputElements);

    uint32_t beginRow = 0;
    uint32_t endRow = 0;
    GetContiguousRange(rowCount,
                       launchBlockDim,
                       beginRow,
                       endRow);
    if (beginRow >= endRow) {
        return;
    }

    const uint32_t paddedOutputBlockWidth =
        kFixedWidthBlock;
    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> outputBlockLocal =
        ubAllocator.Alloc<float>(paddedOutputBlockWidth);
    outputBlockLocal.SetSize(paddedOutputBlockWidth);

    for (uint32_t row = beginRow; row < endRow; ++row) {
        for (uint32_t outputColumnBegin = 0;
             outputColumnBegin < outputWidth;
             outputColumnBegin += kFixedWidthBlock) {
            const uint32_t currentWidthBlock =
                MinU32(kFixedWidthBlock,
                       outputWidth - outputColumnBegin);
            const uint32_t currentPaddedOutputBlockWidth =
                RoundUpU32(currentWidthBlock,
                           kFloatAlignment);
            const uint32_t base =
                row * paddedOutputWidth +
                outputColumnBegin;
            DataCopy(outputBlockLocal,
                     intermediateGm[base],
                     currentPaddedOutputBlockWidth);
            PipeBarrier<PIPE_ALL>();
            Maxs(outputBlockLocal,
                 outputBlockLocal,
                 0.0f,
                 currentPaddedOutputBlockWidth);
            PipeBarrier<PIPE_ALL>();
            DataCopy(outputGm[base],
                     outputBlockLocal,
                     currentPaddedOutputBlockWidth);
            PipeBarrier<PIPE_ALL>();
        }
    }
}



### 4.4 融合优化实验方案：环形缓冲与片上ReLU

`conv_head_fused_ring_relu`使用`ringLocal`保存当前宽度块所需的$K$条输入行块。首次处理一个宽度块时建立窗口，输出行向下移动后仅搬入新进入的一行，并循环覆盖最旧槽位。

卷积结果保留在`outputBlockLocal`中，随后调用与基线完全相同的`Maxs`语句完成ReLU，只把最终结果写回GM。融合方案不需要GM中间缓冲，也不启动第二个ReLU Kernel。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/op_kernel/conv_head_static_tensor.cpp
extern "C" __global__ __aicore__ void conv_head_fused_ring_relu(
    GM_ADDR input,
    GM_ADDR weight,
    GM_ADDR bias,
    GM_ADDR output,
    uint32_t inputChannels,
    uint32_t outputChannels,
    uint32_t inputHeight,
    uint32_t inputWidth,
    uint32_t outputHeight,
    uint32_t outputWidth,
    uint32_t kernelSize,
    uint32_t padding,
    uint32_t paddedInputWidth,
    uint32_t paddedOutputWidth,
    uint32_t paddedKernelWidth,
    uint32_t paddedOutputChannels,
    uint32_t paddedRingWidth,
    uint32_t launchBlockDim) {
    InitSocState();

    if (!CommonArgumentsAreValid(inputChannels,
                                 outputChannels,
                                 inputHeight,
                                 inputWidth,
                                 outputHeight,
                                 outputWidth,
                                 kernelSize,
                                 paddedInputWidth,
                                 paddedOutputWidth,
                                 paddedKernelWidth,
                                 paddedOutputChannels) ||
        paddedRingWidth == 0 ||
        (paddedRingWidth % kFloatAlignment) != 0) {
        return;
    }

    const uint32_t paddedOutputBlockWidth =
        kFixedWidthBlock;
    const uint32_t ringElements =
        inputChannels * kernelSize * paddedRingWidth;
    const uint32_t weightPerOutputChannel =
        inputChannels * kernelSize * paddedKernelWidth;
    const uint32_t ubElements =
        ringElements + weightPerOutputChannel +
        paddedOutputChannels + paddedOutputBlockWidth;
    if (ringElements > kMaxRingElements ||
        weightPerOutputChannel > kMaxWeightElements ||
        ubElements > kMaxUbFloatElements) {
        return;
    }

    const uint32_t inputElements =
        inputChannels * inputHeight * paddedInputWidth;
    const uint32_t weightElements =
        outputChannels * weightPerOutputChannel;
    const uint32_t outputElements =
        outputChannels * outputHeight * paddedOutputWidth;

    GlobalTensor<float> inputGm;
    GlobalTensor<float> weightGm;
    GlobalTensor<float> biasGm;
    GlobalTensor<float> outputGm;
    inputGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(input),
        inputElements);
    weightGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(weight),
        weightElements);
    biasGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(bias),
        paddedOutputChannels);
    outputGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float*>(output),
        outputElements);

    uint32_t beginOutputChannel = 0;
    uint32_t endOutputChannel = 0;
    GetContiguousRange(outputChannels,
                       launchBlockDim,
                       beginOutputChannel,
                       endOutputChannel);
    if (beginOutputChannel >= endOutputChannel) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> ringLocal =
        ubAllocator.Alloc<float>(ringElements);
    LocalTensor<float> weightLocal =
        ubAllocator.Alloc<float>(weightPerOutputChannel);
    LocalTensor<float> biasLocal =
        ubAllocator.Alloc<float>(paddedOutputChannels);
    LocalTensor<float> outputBlockLocal =
        ubAllocator.Alloc<float>(paddedOutputBlockWidth);
    ringLocal.SetSize(ringElements);
    weightLocal.SetSize(weightPerOutputChannel);
    biasLocal.SetSize(paddedOutputChannels);
    outputBlockLocal.SetSize(paddedOutputBlockWidth);

    DataCopy(biasLocal, biasGm, paddedOutputChannels);
    PipeBarrier<PIPE_ALL>();

    for (uint32_t outputChannel = beginOutputChannel;
         outputChannel < endOutputChannel;
         ++outputChannel) {
        DataCopy(weightLocal,
                 weightGm[outputChannel *
                          weightPerOutputChannel],
                 weightPerOutputChannel);
        PipeBarrier<PIPE_ALL>();
        const float channelBias =
            biasLocal.GetValue(outputChannel);

        for (uint32_t outputColumnBegin = 0;
             outputColumnBegin < outputWidth;
             outputColumnBegin += kFixedWidthBlock) {
            const uint32_t currentWidthBlock =
                MinU32(kFixedWidthBlock,
                       outputWidth - outputColumnBegin);
            const uint32_t currentPaddedOutputBlockWidth =
                RoundUpU32(currentWidthBlock,
                           kFloatAlignment);
            uint32_t alignedInputBegin = 0;
            uint32_t copyElements = 0;
            GetAlignedInputBlockRange(outputColumnBegin,
                                      currentWidthBlock,
                                      inputWidth,
                                      kernelSize,
                                      padding,
                                      alignedInputBegin,
                                      copyElements);
            if (copyElements > paddedRingWidth) {
                return;
            }

            // Initialize K row blocks. When the convolution window
            // moves down, only the newly entering block replaces the
            // oldest block in the cyclic LocalTensor buffer.
            for (uint32_t kernelRow = 0;
                 kernelRow < kernelSize;
                 ++kernelRow) {
                const int32_t inputRow =
                    static_cast<int32_t>(kernelRow) -
                    static_cast<int32_t>(padding);
                LoadInputRowBlock(ringLocal,
                                  inputGm,
                                  inputRow,
                                  kernelRow,
                                  alignedInputBegin,
                                  copyElements,
                                  inputChannels,
                                  inputHeight,
                                  paddedInputWidth,
                                  paddedRingWidth);
            }

            uint32_t ringStart = 0;
            for (uint32_t outputRow = 0;
                 outputRow < outputHeight;
                 ++outputRow) {
                ComputeConvolutionBlock(
                    outputBlockLocal,
                    ringLocal,
                    weightLocal,
                    channelBias,
                    ringStart,
                    alignedInputBegin,
                    outputColumnBegin,
                    currentWidthBlock,
                    inputChannels,
                    inputWidth,
                    paddedRingWidth,
                    currentPaddedOutputBlockWidth,
                    kernelSize,
                    paddedKernelWidth,
                    padding);

                // Use exactly the same ReLU primitive, operands and
                // block length as the baseline activation Kernel.
                Maxs(outputBlockLocal,
                     outputBlockLocal,
                     0.0f,
                     currentPaddedOutputBlockWidth);
                PipeBarrier<PIPE_ALL>();
                const uint32_t outputBase =
                    (outputChannel * outputHeight +
                     outputRow) *
                        paddedOutputWidth +
                    outputColumnBegin;
                DataCopy(outputGm[outputBase],
                         outputBlockLocal,
                         currentPaddedOutputBlockWidth);

                if (outputRow + 1 < outputHeight) {
                    const int32_t enteringInputRow =
                        static_cast<int32_t>(outputRow) +
                        static_cast<int32_t>(kernelSize) -
                        static_cast<int32_t>(padding);
                    LoadInputRowBlock(ringLocal,
                                      inputGm,
                                      enteringInputRow,
                                      ringStart,
                                      alignedInputBegin,
                                      copyElements,
                                      inputChannels,
                                      inputHeight,
                                      paddedInputWidth,
                                      paddedRingWidth);
                    ++ringStart;
                    if (ringStart == kernelSize) {
                        ringStart = 0;
                    }
                } else {
                    PipeBarrier<PIPE_ALL>();
                }
            }
        }
    }
}


---
## 5. 结果验证与性能分析

### 5.1 参数解析、ACL资源管理与Host参考结果

Host程序解析设备编号、输入通道、输出通道、空间尺寸、卷积核大小、padding、blockDim、实验方案、预热次数、重复次数和随机种子。`DeviceBuffer`与`StreamGuard`使用析构函数统一释放Device内存和stream，异常路径也能完成资源清理。

固定随机种子分别生成输入、权重和偏置，`convolution_relu_reference`按照实验公式计算CPU参考结果。该参考计算只承担正确性校验，不参与Device侧两种实验方案的性能计时。


In [ ]:
%%writefile Source/04.02/ascend_ops/host_launch/conv_head_npu_main.cpp
#include <acl/acl.h>
#include <aclrtlaunch_conv_head_baseline_conv.h>
#include <aclrtlaunch_conv_head_baseline_relu.h>
#include <aclrtlaunch_conv_head_fused_ring_relu.h>

#include "conv_head_common.h"

#include <algorithm>
#include <cstdint>
#include <cstdlib>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expression)                                                        \
    do {                                                                             \
        const aclError acl_result = (expression);                                    \
        if (acl_result != ACL_SUCCESS) {                                             \
            throw std::runtime_error(                                                \
                std::string("ACL error: ") + #expression +                         \
                ", code=" +                                                         \
                std::to_string(static_cast<int>(acl_result)));                       \
        }                                                                            \
    } while (0)

namespace {

struct NpuConfig : conv_head::Config {
    int32_t device = 0;
    uint32_t warmup = 2;
    uint32_t repeat = 10;
};

struct DeviceBuffer {
    void* ptr = nullptr;

    DeviceBuffer() = default;
    DeviceBuffer(const DeviceBuffer&) = delete;
    DeviceBuffer& operator=(const DeviceBuffer&) = delete;

    ~DeviceBuffer() {
        if (ptr != nullptr) {
            (void)aclrtFree(ptr);
        }
    }
};

struct StreamGuard {
    aclrtStream stream = nullptr;

    StreamGuard() = default;
    StreamGuard(const StreamGuard&) = delete;
    StreamGuard& operator=(const StreamGuard&) = delete;

    ~StreamGuard() {
        if (stream != nullptr) {
            (void)aclrtDestroyStream(stream);
        }
    }
};

struct EventGuard {
    aclrtEvent event = nullptr;

    EventGuard() = default;
    EventGuard(const EventGuard&) = delete;
    EventGuard& operator=(const EventGuard&) = delete;

    ~EventGuard() {
        if (event != nullptr) {
            (void)aclrtDestroyEvent(event);
        }
    }
};

struct RunResult {
    double kernel_us = 0.0;
    conv_head::Metrics metrics;
};

void print_usage(const char* executable) {
    std::cout
        << "Usage: " << executable << " [options]\n"
        << "NCHW FP32 convolution + ReLU, batch=1, stride=1.\n\n"
        << "Options:\n"
        << "  --device <id>       device id, default: 0\n"
        << "  --cin <num>         input channels, default: 8\n"
        << "  --cout <num>        output channels, default: 16\n"
        << "  --height <num>      input height, default: 64\n"
        << "  --width <num>       input width, default: 64\n"
        << "  --kernel <num>      square kernel size, default: 3\n"
        << "  --padding <num>     zero padding, default: 1\n"
        << "  --block-dim <num>   AI Core launch blockDim, default: 8\n"
        << "  --version <name>    basic/fused/all, default: all\n"
        << "  --warmup <num>      warmup count, default: 2\n"
        << "  --repeat <num>      measured repeats, default: 10\n"
        << "  --seed <num>        fixed random seed, default: 1234\n"
        << "  --sweep             test H=W=32/64/96/128\n"
        << "  --print-output      print the first 16 output values\n"
        << "  -h, --help          show this help\n";
}

NpuConfig parse_arguments(int argc, char** argv) {
    NpuConfig cfg;
    for (int index = 1; index < argc; ++index) {
        const std::string argument = argv[index];
        auto value_after = [&](const std::string& option) -> const char* {
            if (index + 1 >= argc) {
                throw std::invalid_argument(
                    "missing value after " + option);
            }
            return argv[++index];
        };

        if (argument == "--device") {
            cfg.device = std::stoi(value_after(argument));
        } else if (argument == "--cin") {
            cfg.input_channels = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--cout") {
            cfg.output_channels = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--height") {
            cfg.input_height = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--width") {
            cfg.input_width = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--kernel") {
            cfg.kernel_size = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--padding") {
            cfg.padding = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--block-dim") {
            cfg.block_dim = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--version") {
            cfg.version =
                conv_head::parse_version(value_after(argument));
        } else if (argument == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--seed") {
            cfg.seed = static_cast<uint32_t>(
                std::stoul(value_after(argument)));
        } else if (argument == "--sweep") {
            cfg.sweep = true;
        } else if (argument == "--print-output") {
            cfg.print_output = true;
        } else if (argument == "-h" || argument == "--help") {
            print_usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument(
                "unknown argument: " + argument);
        }
    }

    if (cfg.repeat == 0) {
        throw std::invalid_argument("--repeat must be positive");
    }
    conv_head::check_config(cfg);
    return cfg;
}

std::vector<float> make_input(const NpuConfig& cfg) {
    std::mt19937 generator(cfg.seed);
    std::uniform_real_distribution<float> distribution(-1.0f, 1.0f);
    std::vector<float> input(
        static_cast<size_t>(conv_head::valid_input_elements(cfg)));
    for (float& value : input) {
        value = distribution(generator);
    }
    return input;
}

std::vector<float> make_weight(const NpuConfig& cfg) {
    std::mt19937 generator(cfg.seed + 17u);
    std::uniform_real_distribution<float> distribution(-0.25f, 0.25f);
    std::vector<float> weight(
        static_cast<size_t>(conv_head::valid_weight_elements(cfg)));
    for (float& value : weight) {
        value = distribution(generator);
    }
    return weight;
}

std::vector<float> make_bias(const NpuConfig& cfg) {
    std::mt19937 generator(cfg.seed + 29u);
    std::uniform_real_distribution<float> distribution(-0.1f, 0.1f);
    std::vector<float> bias(cfg.output_channels);
    for (float& value : bias) {
        value = distribution(generator);
    }
    return bias;
}

std::vector<float> convolution_relu_reference(
    const NpuConfig& cfg,
    const std::vector<float>& input,
    const std::vector<float>& weight,
    const std::vector<float>& bias) {
    const uint32_t outputHeight =
        conv_head::output_extent(cfg.input_height,
                                 cfg.kernel_size,
                                 cfg.padding);
    const uint32_t outputWidth =
        conv_head::output_extent(cfg.input_width,
                                 cfg.kernel_size,
                                 cfg.padding);
    std::vector<float> output(
        static_cast<size_t>(conv_head::valid_output_elements(cfg)),
        0.0f);

    for (uint32_t outputChannel = 0;
         outputChannel < cfg.output_channels;
         ++outputChannel) {
        for (uint32_t outputRow = 0;
             outputRow < outputHeight;
             ++outputRow) {
            for (uint32_t outputColumn = 0;
                 outputColumn < outputWidth;
                 ++outputColumn) {
                double accumulator =
                    static_cast<double>(bias[outputChannel]);
                for (uint32_t inputChannel = 0;
                     inputChannel < cfg.input_channels;
                     ++inputChannel) {
                    for (uint32_t kernelRow = 0;
                         kernelRow < cfg.kernel_size;
                         ++kernelRow) {
                        const int64_t inputRow =
                            static_cast<int64_t>(outputRow) +
                            kernelRow - cfg.padding;
                        if (inputRow < 0 ||
                            inputRow >= cfg.input_height) {
                            continue;
                        }
                        for (uint32_t kernelColumn = 0;
                             kernelColumn < cfg.kernel_size;
                             ++kernelColumn) {
                            const int64_t inputColumn =
                                static_cast<int64_t>(outputColumn) +
                                kernelColumn - cfg.padding;
                            if (inputColumn < 0 ||
                                inputColumn >= cfg.input_width) {
                                continue;
                            }
                            const uint64_t inputIndex =
                                (static_cast<uint64_t>(inputChannel) *
                                     cfg.input_height +
                                 static_cast<uint32_t>(inputRow)) *
                                    cfg.input_width +
                                static_cast<uint32_t>(inputColumn);
                            const uint64_t weightIndex =
                                ((static_cast<uint64_t>(outputChannel) *
                                      cfg.input_channels +
                                  inputChannel) *
                                     cfg.kernel_size +
                                 kernelRow) *
                                    cfg.kernel_size +
                                kernelColumn;
                            accumulator +=
                                static_cast<double>(input[inputIndex]) *
                                static_cast<double>(weight[weightIndex]);
                        }
                    }
                }
                const uint64_t outputIndex =
                    (static_cast<uint64_t>(outputChannel) *
                         outputHeight +
                     outputRow) *
                        outputWidth +
                    outputColumn;
                output[outputIndex] =
                    accumulator > 0.0
                        ? static_cast<float>(accumulator)
                        : 0.0f;
            }
        }
    }
    return output;
}



### 5.2 补齐数据的准备与有效输出恢复

Host按照32字节对齐要求建立补齐后的输入、权重、偏置和输出。有效输入与卷积核元素复制到每条补齐行的前部，其余位置置0；Device输出回传后，`unpad_output`只提取每条输出行的有效列。

两种实验方案使用完全相同的补齐数据，避免数据布局不同影响性能比较。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/host_launch/conv_head_npu_main.cpp
std::vector<float> pad_input(const NpuConfig& cfg,
                             const std::vector<float>& input,
                             uint32_t paddedInputWidth) {
    const uint64_t paddedCount = conv_head::checked_mul(
        conv_head::checked_mul(cfg.input_channels,
                               cfg.input_height,
                               "padded input count"),
        paddedInputWidth,
        "padded input count");
    std::vector<float> padded(static_cast<size_t>(paddedCount), 0.0f);
    for (uint32_t channel = 0;
         channel < cfg.input_channels;
         ++channel) {
        for (uint32_t row = 0;
             row < cfg.input_height;
             ++row) {
            const uint64_t sourceBase =
                (static_cast<uint64_t>(channel) *
                     cfg.input_height +
                 row) *
                cfg.input_width;
            const uint64_t destinationBase =
                (static_cast<uint64_t>(channel) *
                     cfg.input_height +
                 row) *
                paddedInputWidth;
            std::copy_n(
                input.begin() +
                    static_cast<std::ptrdiff_t>(sourceBase),
                cfg.input_width,
                padded.begin() +
                    static_cast<std::ptrdiff_t>(destinationBase));
        }
    }
    return padded;
}

std::vector<float> pad_weight(const NpuConfig& cfg,
                              const std::vector<float>& weight,
                              uint32_t paddedKernelWidth) {
    const uint64_t paddedCount = conv_head::checked_mul(
        conv_head::checked_mul(cfg.output_channels,
                               cfg.input_channels,
                               "padded weight count"),
        conv_head::checked_mul(cfg.kernel_size,
                               paddedKernelWidth,
                               "padded weight count"),
        "padded weight count");
    std::vector<float> padded(static_cast<size_t>(paddedCount), 0.0f);
    for (uint32_t outputChannel = 0;
         outputChannel < cfg.output_channels;
         ++outputChannel) {
        for (uint32_t inputChannel = 0;
             inputChannel < cfg.input_channels;
             ++inputChannel) {
            for (uint32_t kernelRow = 0;
                 kernelRow < cfg.kernel_size;
                 ++kernelRow) {
                const uint64_t sourceBase =
                    ((static_cast<uint64_t>(outputChannel) *
                          cfg.input_channels +
                      inputChannel) *
                         cfg.kernel_size +
                     kernelRow) *
                    cfg.kernel_size;
                const uint64_t destinationBase =
                    ((static_cast<uint64_t>(outputChannel) *
                          cfg.input_channels +
                      inputChannel) *
                         cfg.kernel_size +
                     kernelRow) *
                    paddedKernelWidth;
                std::copy_n(
                    weight.begin() +
                        static_cast<std::ptrdiff_t>(sourceBase),
                    cfg.kernel_size,
                    padded.begin() +
                        static_cast<std::ptrdiff_t>(
                            destinationBase));
            }
        }
    }
    return padded;
}

std::vector<float> unpad_output(const NpuConfig& cfg,
                                const std::vector<float>& padded,
                                uint32_t outputHeight,
                                uint32_t outputWidth,
                                uint32_t paddedOutputWidth) {
    std::vector<float> output(
        static_cast<size_t>(conv_head::valid_output_elements(cfg)),
        0.0f);
    for (uint32_t channel = 0;
         channel < cfg.output_channels;
         ++channel) {
        for (uint32_t row = 0;
             row < outputHeight;
             ++row) {
            const uint64_t sourceBase =
                (static_cast<uint64_t>(channel) *
                     outputHeight +
                 row) *
                paddedOutputWidth;
            const uint64_t destinationBase =
                (static_cast<uint64_t>(channel) *
                     outputHeight +
                 row) *
                outputWidth;
            std::copy_n(
                padded.begin() +
                    static_cast<std::ptrdiff_t>(sourceBase),
                outputWidth,
                output.begin() +
                    static_cast<std::ptrdiff_t>(
                        destinationBase));
        }
    }
    return output;
}



### 5.3 误差校验与输出示例

`compare_output`逐元素统计最大绝对误差、最大相对误差和超差元素数量。误差判定同时使用绝对误差与相对误差阈值：

$$
|y-ref|>atol+rtol\cdot|ref|
$$

满足该条件的有效输出元素计入`errors`。`--print-output`可额外打印第一个输出通道的前16个结果，用于直观检查。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/host_launch/conv_head_npu_main.cpp
conv_head::Metrics compare_output(
    const std::vector<float>& output,
    const std::vector<float>& reference,
    double absoluteTolerance = 1.0e-3,
    double relativeTolerance = 1.0e-3) {
    if (output.size() != reference.size()) {
        throw std::invalid_argument(
            "output/reference size mismatch");
    }
    conv_head::Metrics metrics;
    for (size_t index = 0; index < output.size(); ++index) {
        const double actual = output[index];
        const double expected = reference[index];
        const double absoluteError =
            std::abs(actual - expected);
        const double relativeError =
            absoluteError /
            std::max(1.0e-6, std::abs(expected));
        metrics.max_abs_error =
            std::max(metrics.max_abs_error, absoluteError);
        metrics.max_rel_error =
            std::max(metrics.max_rel_error, relativeError);
        if (absoluteError >
            absoluteTolerance +
                relativeTolerance * std::abs(expected)) {
            ++metrics.mismatch_count;
        }
    }
    return metrics;
}

void print_output_sample(const NpuConfig& cfg,
                         const std::vector<float>& output,
                         const std::vector<float>& reference) {
    const uint32_t outputWidth =
        conv_head::output_extent(cfg.input_width,
                                 cfg.kernel_size,
                                 cfg.padding);
    const size_t count =
        std::min<size_t>(16, output.size());
    std::cout << "sample (output channel 0, row 0):\n";
    for (size_t index = 0; index < count; ++index) {
        std::cout << "  col=" << (index % outputWidth)
                  << " output=" << output[index]
                  << " reference=" << reference[index]
                  << "\n";
    }
}



### 5.4 Device内存申请、Kernel启动与重复测量

程序为输入、权重、偏置和最终输出申请Device内存。只有基线实验方案额外申请卷积中间缓冲区；融合方案不申请该缓冲。固定输入在正式测量前上传一次，Host到Device传输不计入两种Kernel组织的对比。

`launch_once`根据实验方案启动对应Kernel。基线方案连续启动卷积和ReLU两个Kernel，融合方案启动一个Kernel，每轮结束后同步stream。程序先执行`warmup`次预热，再执行`repeat`次正式测量并计算平均`kernel_us`。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/host_launch/conv_head_npu_main.cpp
RunResult run_variant(const NpuConfig& cfg,
                      conv_head::Version version) {
    if (version != conv_head::Version::Basic &&
        version != conv_head::Version::Fused) {
        throw std::invalid_argument(
            "run_variant requires basic or fused");
    }
    conv_head::check_config(cfg);

    const uint32_t outputHeight =
        conv_head::output_extent(cfg.input_height,
                                 cfg.kernel_size,
                                 cfg.padding);
    const uint32_t outputWidth =
        conv_head::output_extent(cfg.input_width,
                                 cfg.kernel_size,
                                 cfg.padding);
    const uint32_t paddedInputWidth =
        conv_head::padded_width(cfg.input_width);
    const uint32_t paddedOutputWidth =
        conv_head::padded_width(outputWidth);
    const uint32_t paddedKernelWidth =
        conv_head::padded_width(cfg.kernel_size);
    const uint32_t paddedOutputChannels =
        conv_head::padded_width(cfg.output_channels);
    const uint32_t paddedRingWidth =
        conv_head::fixed_ring_row_width(cfg, outputWidth);
    const std::vector<float> input = make_input(cfg);
    const std::vector<float> weight = make_weight(cfg);
    const std::vector<float> bias = make_bias(cfg);
    const std::vector<float> reference =
        convolution_relu_reference(cfg, input, weight, bias);
    const std::vector<float> paddedInput =
        pad_input(cfg, input, paddedInputWidth);
    const std::vector<float> paddedWeight =
        pad_weight(cfg, weight, paddedKernelWidth);
    std::vector<float> paddedBias(paddedOutputChannels, 0.0f);
    std::copy(bias.begin(), bias.end(), paddedBias.begin());

    const uint64_t paddedOutputCount = conv_head::checked_mul(
        conv_head::checked_mul(cfg.output_channels,
                               outputHeight,
                               "padded output count"),
        paddedOutputWidth,
        "padded output count");
    std::vector<float> paddedOutput(
        static_cast<size_t>(paddedOutputCount),
        0.0f);

    const size_t inputBytes =
        conv_head::float_bytes(paddedInput.size());
    const size_t weightBytes =
        conv_head::float_bytes(paddedWeight.size());
    const size_t biasBytes =
        conv_head::float_bytes(paddedBias.size());
    const size_t outputBytes =
        conv_head::float_bytes(paddedOutput.size());

    DeviceBuffer inputDevice;
    DeviceBuffer weightDevice;
    DeviceBuffer biasDevice;
    DeviceBuffer intermediateDevice;
    DeviceBuffer outputDevice;
    StreamGuard stream;

    ACL_CHECK(aclrtCreateStream(&stream.stream));
    ACL_CHECK(aclrtMalloc(&inputDevice.ptr,
                          inputBytes,
                          ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&weightDevice.ptr,
                          weightBytes,
                          ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&biasDevice.ptr,
                          biasBytes,
                          ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&outputDevice.ptr,
                          outputBytes,
                          ACL_MEM_MALLOC_HUGE_FIRST));
    if (version == conv_head::Version::Basic) {
        ACL_CHECK(aclrtMalloc(&intermediateDevice.ptr,
                              outputBytes,
                              ACL_MEM_MALLOC_HUGE_FIRST));
    }

    ACL_CHECK(aclrtMemcpy(inputDevice.ptr,
                          inputBytes,
                          paddedInput.data(),
                          inputBytes,
                          ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(weightDevice.ptr,
                          weightBytes,
                          paddedWeight.data(),
                          weightBytes,
                          ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(biasDevice.ptr,
                          biasBytes,
                          paddedBias.data(),
                          biasBytes,
                          ACL_MEMCPY_HOST_TO_DEVICE));

    auto launch_baseline_conv = [&]() {
        ACLRT_LAUNCH_KERNEL(conv_head_baseline_conv)(
                cfg.block_dim,
                stream.stream,
                inputDevice.ptr,
                weightDevice.ptr,
                biasDevice.ptr,
                intermediateDevice.ptr,
                cfg.input_channels,
                cfg.output_channels,
                cfg.input_height,
                cfg.input_width,
                outputHeight,
                outputWidth,
                cfg.kernel_size,
                cfg.padding,
                paddedInputWidth,
                paddedOutputWidth,
                paddedKernelWidth,
                paddedOutputChannels,
                paddedRingWidth,
                cfg.block_dim);
    };

    auto launch_baseline_relu = [&]() {
        ACLRT_LAUNCH_KERNEL(conv_head_baseline_relu)(
                cfg.block_dim,
                stream.stream,
                intermediateDevice.ptr,
                outputDevice.ptr,
                cfg.output_channels,
                outputHeight,
                outputWidth,
                paddedOutputWidth,
                cfg.block_dim);
    };

    auto launch_fused = [&]() {
        ACLRT_LAUNCH_KERNEL(conv_head_fused_ring_relu)(
                cfg.block_dim,
                stream.stream,
                inputDevice.ptr,
                weightDevice.ptr,
                biasDevice.ptr,
                outputDevice.ptr,
                cfg.input_channels,
                cfg.output_channels,
                cfg.input_height,
                cfg.input_width,
                outputHeight,
                outputWidth,
                cfg.kernel_size,
                cfg.padding,
                paddedInputWidth,
                paddedOutputWidth,
                paddedKernelWidth,
                paddedOutputChannels,
                paddedRingWidth,
                cfg.block_dim);
    };

    auto launch_once = [&]() {
        if (version == conv_head::Version::Basic) {
            launch_baseline_conv();
            launch_baseline_relu();
        } else {
            launch_fused();
        }
        ACL_CHECK(aclrtSynchronizeStream(stream.stream));
    };

    auto create_timeline_event = [](EventGuard& event) {
        ACL_CHECK(aclrtCreateEventExWithFlag(
            &event.event,
            ACL_EVENT_TIME_LINE));
    };

    auto elapsed_us = [](const EventGuard& start,
                         const EventGuard& end) -> double {
        float elapsedMs = 0.0f;
        ACL_CHECK(aclrtEventElapsedTime(
            &elapsedMs,
            start.event,
            end.event));
        return static_cast<double>(elapsedMs) * 1000.0;
    };

    auto measure_kernel_us_once = [&]() -> double {
        EventGuard firstStart;
        EventGuard firstEnd;
        create_timeline_event(firstStart);
        create_timeline_event(firstEnd);

        ACL_CHECK(aclrtRecordEvent(firstStart.event, stream.stream));
        if (version == conv_head::Version::Basic) {
            launch_baseline_conv();
        } else {
            launch_fused();
        }
        ACL_CHECK(aclrtRecordEvent(firstEnd.event, stream.stream));

        EventGuard secondStart;
        EventGuard secondEnd;
        if (version == conv_head::Version::Basic) {
            create_timeline_event(secondStart);
            create_timeline_event(secondEnd);
            ACL_CHECK(aclrtRecordEvent(secondStart.event, stream.stream));
            launch_baseline_relu();
            ACL_CHECK(aclrtRecordEvent(secondEnd.event, stream.stream));
        }

        ACL_CHECK(aclrtSynchronizeStream(stream.stream));

        double kernelUs = elapsed_us(firstStart, firstEnd);
        if (version == conv_head::Version::Basic) {
            kernelUs += elapsed_us(secondStart, secondEnd);
        }
        return kernelUs;
    };

    for (uint32_t iteration = 0;
         iteration < cfg.warmup;
         ++iteration) {
        launch_once();
    }

    double accumulatedUs = 0.0;
    for (uint32_t iteration = 0;
         iteration < cfg.repeat;
         ++iteration) {
        accumulatedUs += measure_kernel_us_once();
    }

    ACL_CHECK(aclrtMemcpy(paddedOutput.data(),
                          outputBytes,
                          outputDevice.ptr,
                          outputBytes,
                          ACL_MEMCPY_DEVICE_TO_HOST));
    const std::vector<float> output =
        unpad_output(cfg,
                     paddedOutput,
                     outputHeight,
                     outputWidth,
                     paddedOutputWidth);

    RunResult result;
    result.kernel_us =
        accumulatedUs / static_cast<double>(cfg.repeat);
    result.metrics = compare_output(output, reference);
    conv_head::print_result(cfg,
                            version,
                            result.kernel_us,
                            result.metrics);

    if (cfg.print_output) {
        print_output_sample(cfg, output, reference);
    }
    return result;
}



### 5.5 结果回传、指标输出与主流程

正式测量结束后，Host回传最终输出并与参考结果比较。输出字段包括卷积尺寸、估算输入搬入次数、实验方案、平均执行时间、估算读写量、有效带宽、误差、错误数量和状态。

当命令参数为`all`时，主流程依次运行`basic`与`fused`并输出加速比；`--sweep`用于按若干空间尺寸运行输入规模扩展实验。


In [ ]:
%%writefile -a Source/04.02/ascend_ops/host_launch/conv_head_npu_main.cpp
void run_case(const NpuConfig& cfg) {
    if (cfg.version == conv_head::Version::Basic) {
        (void)run_variant(cfg, conv_head::Version::Basic);
        return;
    }
    if (cfg.version == conv_head::Version::Fused) {
        (void)run_variant(cfg, conv_head::Version::Fused);
        return;
    }

    const RunResult basic =
        run_variant(cfg, conv_head::Version::Basic);
    const RunResult fused =
        run_variant(cfg, conv_head::Version::Fused);
    if (fused.kernel_us > 0.0) {
        std::cout << "  fused speedup over basic: "
                  << std::fixed
                  << std::setprecision(3)
                  << basic.kernel_us / fused.kernel_us
                  << "x\n";
    }
}

}  // namespace

int main(int argc, char** argv) {
    NpuConfig cfg;
    bool aclInitialized = false;
    bool deviceSet = false;

    auto cleanup = [&]() noexcept {
        if (deviceSet) {
            (void)aclrtResetDevice(cfg.device);
            deviceSet = false;
        }
        if (aclInitialized) {
            (void)aclFinalize();
            aclInitialized = false;
        }
    };

    try {
        cfg = parse_arguments(argc, argv);
        ACL_CHECK(aclInit(nullptr));
        aclInitialized = true;
        ACL_CHECK(aclrtSetDevice(cfg.device));
        deviceSet = true;

        conv_head::print_header();
        if (cfg.sweep) {
            for (uint32_t extent : {32u, 64u, 96u, 128u}) {
                NpuConfig sweepConfig = cfg;
                sweepConfig.input_height = extent;
                sweepConfig.input_width = extent;
                conv_head::check_config(sweepConfig);
                run_case(sweepConfig);
            }
        } else {
            run_case(cfg);
        }

        cleanup();
        return 0;
    } catch (const std::exception& error) {
        cleanup();
        std::cerr << "error: " << error.what() << "\n";
        print_usage(argv[0]);
        return 1;
    }
}


### 5.6 Kernel启动头文件示例

Ascend C构建过程会根据三个Device入口生成对应的`aclrtlaunch_*.h`。下面的小文件集中展示Host侧需要包含的自动生成头文件，可用于核对Kernel名称。


In [ ]:
%%writefile Source/04.02/ascend_ops/host_launch/conv_head_launch_example.cpp
#include <aclrtlaunch_conv_head_baseline_conv.h>
#include <aclrtlaunch_conv_head_baseline_relu.h>
#include <aclrtlaunch_conv_head_fused_ring_relu.h>


### 5.7 工程构建

CMake通过`ascendc_library`编译Device核函数，再编译NPU Host程序并链接`ascendcl`。工程仅生成NPU实验可执行程序，Host参考计算直接包含在主程序中。

默认`SOC_VERSION`为`ascend910b1`，与实验2.1的设置保持一致；如果目标环境要求其他编译型号，可通过运行脚本的`-v`参数覆盖。


In [ ]:
%%writefile Source/04.02/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ascendc_static_tensor_conv_head LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build the Ascend C NPU experiment" ON)

if(NOT BUILD_ASCEND)
  message(FATAL_ERROR
    "This experiment intentionally has no CPU simulation target. "
    "Configure with -DBUILD_ASCEND=ON in a CANN environment.")
endif()

set(RUN_MODE "npu" CACHE STRING
    "Ascend C toolchain run mode; use npu for this experiment")
set(SOC_VERSION "ascend910b1" CACHE STRING
    "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH
    "CANN installation path")
if(NOT ASCEND_CANN_PATH)
  set(ASCEND_CANN_PATH
      "/usr/local/Ascend/ascend-toolkit/latest"
      CACHE PATH "CANN installation path" FORCE)
endif()
set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH
    "CANN package path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH
    "Ascend C install output" FORCE)

if(EXISTS
   "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE
      "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
elseif(EXISTS
       "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  set(ASCENDC_CMAKE_FILE
      "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
else()
  message(FATAL_ERROR
    "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. "
    "Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
endif()

message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
message(STATUS "SOC_VERSION=${SOC_VERSION}")
include("${ASCENDC_CMAKE_FILE}")

ascendc_library(conv_head_kernels STATIC
    ascend_ops/op_kernel/conv_head_static_tensor.cpp
)
ascendc_compile_definitions(conv_head_kernels PRIVATE
    -DASCENDC_DUMP=0
)

add_executable(conv_head_ascend_demo
    ascend_ops/host_launch/conv_head_npu_main.cpp
)
target_include_directories(conv_head_ascend_demo PRIVATE
    include
    ${ASCEND_CANN_PACKAGE_PATH}/include
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
    ${CMAKE_INSTALL_PREFIX}/include/conv_head_kernels
    ${CMAKE_BINARY_DIR}/out/include/conv_head_kernels
)
target_link_directories(conv_head_ascend_demo PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/lib64
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
    ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
)
target_link_libraries(conv_head_ascend_demo PRIVATE
    conv_head_kernels
    ascendcl
)
target_compile_options(conv_head_ascend_demo PRIVATE
    -Wall
    -Wextra
)
add_dependencies(conv_head_ascend_demo conv_head_kernels)

install(TARGETS conv_head_ascend_demo RUNTIME DESTINATION bin)
install(DIRECTORY ascend_ops
        DESTINATION share/ascendc_static_tensor_conv_head)


### 5.8 构建、运行与Profiling脚本

运行脚本负责定位CANN、配置CMake、编译工程并传递实验参数。为避免CMake增量编译复用上次残留的目标文件，下面的实验命令统一加入`-c`，每次运行前清理`build_ascend`后重新配置和编译。

Profiling脚本接收`basic`或`fused`作为第一个参数，并使用本Notebook主实验相同的输入规模、卷积参数和运行配置。


In [ ]:
%%writefile Source/04.02/scripts/run_ascend.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"

resolve_ascend_install_path() {
  if [[ -n "${ASCEND_INSTALL_PATH}" && -d "${ASCEND_INSTALL_PATH}" ]]; then
    return 0
  fi

  local candidates=()
  [[ -n "${ASCEND_TOOLKIT_HOME:-}" ]] && candidates+=("${ASCEND_TOOLKIT_HOME}")
  [[ -n "${ASCEND_HOME_PATH:-}" ]] && candidates+=("${ASCEND_HOME_PATH}")
  [[ -n "${ASCEND_CANN_PACKAGE_PATH:-}" ]] && candidates+=("${ASCEND_CANN_PACKAGE_PATH}")
  candidates+=(
    "${ASCEND_INSTALL_PATH_DEFAULT}"
    "/usr/local/Ascend/ascend-toolkit"
    "/opt/Ascend/ascend-toolkit/latest"
    "/opt/Ascend/ascend-toolkit"
    "${HOME}/Ascend/ascend-toolkit/latest"
    "${HOME}/Ascend/ascend-toolkit"
    "/workspace/Ascend/ascend-toolkit/latest"
    "/workspace/Ascend/ascend-toolkit"
  )

  local path
  for path in "${candidates[@]}"; do
    if [[ -n "${path}" && -d "${path}" ]]; then
      ASCEND_INSTALL_PATH="${path}"
      return 0
    fi
  done

  local found_set_env=""
  found_set_env=$(find /usr/local/Ascend /opt/Ascend "${HOME}" /workspace \
    -path "*/ascend-toolkit*/set_env.sh" -print -quit 2>/dev/null || true)
  if [[ -n "${found_set_env}" ]]; then
    ASCEND_INSTALL_PATH="$(dirname "${found_set_env}")"
    return 0
  fi
  return 1
}

DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
CIN=8
COUT=16
HEIGHT=64
WIDTH=64
KERNEL=3
PADDING=1
BLOCK_DIM=8
VERSION="all"
WARMUP=2
REPEAT=10
SEED=1234
SWEEP=0
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run_ascend.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH; auto-detected when omitted
  -v <soc>    SOC_VERSION, default: ascend910b1 (same as experiment 2.1)
  -d <id>     device id, default: 0
  -C <num>    input channels, default: 8
  -O <num>    output channels, default: 16
  -H <num>    input height, default: 64
  -W <num>    input width, default: 64
  -K <num>    square kernel size, default: 3
  -P <num>    zero padding, default: 1
  -b <num>    AI Core launch blockDim, default: 8
  -V <name>   basic, fused, or all; default: all
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 10
  -e <num>    random seed, default: 1234
  -m <mode>   CMake run mode, default: npu
  -t <type>   CMake build type, default: Release
  -s          sweep H=W over 32/64/96/128
  -c          remove build_ascend before building
  -h          show help

Examples:
  bash scripts/run_ascend.sh -c
  bash scripts/run_ascend.sh -c -V basic
  bash scripts/run_ascend.sh -c -V fused
  bash scripts/run_ascend.sh -c -C 8 -O 32 -H 128 -W 128 -K 3 -P 1 -s
  bash scripts/run_ascend.sh -c -- --print-output
USAGE
}

while getopts ":a:v:d:C:O:H:W:K:P:b:V:w:r:e:m:t:sch" opt; do
  case "${opt}" in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    C) CIN="${OPTARG}" ;;
    O) COUT="${OPTARG}" ;;
    H) HEIGHT="${OPTARG}" ;;
    W) WIDTH="${OPTARG}" ;;
    K) KERNEL="${OPTARG}" ;;
    P) PADDING="${OPTARG}" ;;
    b) BLOCK_DIM="${OPTARG}" ;;
    V) VERSION="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    e) SEED="${OPTARG}" ;;
    m) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    s) SWEEP=1 ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if ! resolve_ascend_install_path; then
  echo "Cannot find ascend-toolkit. Set ASCEND_INSTALL_PATH or pass -a <path>." >&2
  exit 1
fi
if [[ ! -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  echo "Cannot find set_env.sh under ${ASCEND_INSTALL_PATH}." >&2
  exit 1
fi

source "${ASCEND_INSTALL_PATH}/set_env.sh"
export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi

cmake -S "${SCRIPT_DIR}" -B "${BUILD_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"
cmake --build "${BUILD_DIR}" -j

BIN="${BUILD_DIR}/conv_head_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CMD=(
  "${BIN}"
  --device "${DEVICE_ID}"
  --cin "${CIN}"
  --cout "${COUT}"
  --height "${HEIGHT}"
  --width "${WIDTH}"
  --kernel "${KERNEL}"
  --padding "${PADDING}"
  --block-dim "${BLOCK_DIM}"
  --version "${VERSION}"
  --warmup "${WARMUP}"
  --repeat "${REPEAT}"
  --seed "${SEED}"
)
if [[ "${SWEEP}" == "1" ]]; then
  CMD+=(--sweep)
fi
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
%%writefile Source/04.02/scripts/profile_msprof_template.sh
#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
VERSION="${1:-fused}"
BIN="${ROOT}/build_ascend/conv_head_ascend_demo"
OUT_DIR="${ROOT}/profile_${VERSION}_$(date +%Y%m%d_%H%M%S)"

if [[ ! -x "${BIN}" ]]; then
  echo "Build the experiment first with scripts/run_ascend.sh." >&2
  exit 1
fi
if [[ "${VERSION}" != "basic" && "${VERSION}" != "fused" ]]; then
  echo "Profile version must be basic or fused." >&2
  exit 1
fi

msprof \
  --application="${BIN} --device 0 --cin 8 --cout 16 --height 64 --width 64 --kernel 3 --padding 1 --block-dim 8 --version ${VERSION} --warmup 5 --repeat 50 --seed 1234" \
  --output="${OUT_DIR}"


In [ ]:
!chmod +x Source/04.02/scripts/run_ascend.sh
!chmod +x Source/04.02/scripts/profile_msprof_template.sh
!find Source/04.02 -maxdepth 3 -type f | sort


### 5.9 实验参数与完整运行

| 参数 | 数值 | 参数意义 |
|---|---:|---|
| 输入规模Cin,Cout | 4,16 | 输入通道数和输出通道数 |
| 输入空间H,W | 32,32 | 输入特征图高度和宽度 |
| 卷积核K | 3 | 方形卷积核边长 |
| padding | 1 | 输入四周零填充宽度 |
| stride | 1 | 当前实现固定步长 |
| blockDim | 8 | Kernel启动逻辑任务块数量 |
| warmup | 2 | 正式计时前的预运行次数 |
| repeat | 10 | 正式计时的重复次数 |
| seed | 1234 | 保证两种实验方案使用相同输入 |

下面先清理并编译一次工程，再分别运行基线实验方案和融合优化实验方案。每种方案的完整终端输出写入`results`目录。


In [ ]:
%%bash
set -euo pipefail
cd Source/04.02
mkdir -p results

bash scripts/run_ascend.sh -c -C 8 -O 16 -H 64 -W 64 -K 3 -P 1 -b 8 -V basic -w 2 -r 10 -e 1234 \
  | tee results/basic.log
bash scripts/run_ascend.sh -c -C 8 -O 16 -H 64 -W 64 -K 3 -P 1 -b 8 -V fused -w 2 -r 10 -e 1234 \
  | tee results/fused.log


### 5.10 汇总运行结果

下面从两个日志中提取结果行，生成统一CSV并计算融合版相对基线的加速比。先检查`status`和`errors`，只有两个实验方案均为PASS时，性能比较才有意义。


In [ ]:
import csv
from pathlib import Path
from IPython.display import Markdown, display

result_dir = Path("Source/04.02/results")
versions = ("basic", "fused")
rows = []

for version in versions:
    log_path = result_dir / f"{version}.log"
    if not log_path.exists():
        print("Missing log:", log_path)
        continue
    text = log_path.read_text(encoding="utf-8", errors="replace")
    data = None
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 16 and parts[7] == version:
            data = {
                "Cin": int(parts[0]),
                "Cout": int(parts[1]),
                "H": int(parts[2]),
                "W": int(parts[3]),
                "K": int(parts[4]),
                "pad": int(parts[5]),
                "inputCopies": int(parts[6]),
                "version": parts[7],
                "kernel_us": float(parts[8]),
                "read_MiB": float(parts[9]),
                "write_MiB": float(parts[10]),
                "GB/s": float(parts[11]),
                "max_abs": float(parts[12]),
                "max_rel": float(parts[13]),
                "errors": int(parts[14]),
                "status": parts[15],
            }
            break
    if data is None:
        print("No result row found in:", log_path)
        continue
    rows.append(data)

if rows:
    csv_path = result_dir / "conv_head_results.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    headers = [
        "version", "inputCopies", "kernel_us", "read_MiB", "write_MiB",
        "GB/s", "max_abs", "max_rel", "errors", "status",
    ]
    table = ["|" + "|".join(headers) + "|", "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        table.append("|" + "|".join(str(row[name]) for name in headers) + "|")
    display(Markdown("\n".join(table)))

    by_version = {row["version"]: row for row in rows}
    if all(name in by_version for name in versions):
        basic = by_version["basic"]["kernel_us"]
        fused = by_version["fused"]["kernel_us"]
        print(f"basic -> fused speedup: {basic / fused:.3f}x")
        print(
            "inputCopies reduction:",
            f"{(by_version['basic']['inputCopies'] - by_version['fused']['inputCopies']) * 100 / by_version['basic']['inputCopies']:.2f}%",
        )
        print(
            "All versions PASS:",
            all(row["status"] == "PASS" and row["errors"] == 0 for row in rows),
        )
    print("CSV saved to:", csv_path)


### 5.11 结果判读与性能分析

运行完成后可按以下顺序查看结果：

1. **正确性指标**：查看`errors`和`status`，确认两个实验方案是否通过校验；同时结合`max_abs`和`max_rel`判断Device输出与Host参考结果之间的数值误差。
2. **执行时间**：比较两个实验方案的`kernel_us`。基线方案的时间覆盖卷积Kernel和独立激活Kernel，融合方案的时间覆盖融合Kernel。
3. **加速比**：使用`kernel_us_basic / kernel_us_fused`计算融合方案相对基线的加速比。加速比大于1表示融合方案执行时间更短，小于1表示本次配置下没有获得加速。
4. **输入数据复用**：比较`inputCopies`和`read_MiB`，观察环形行缓冲区是否减少相邻卷积窗口的重复输入搬入。
5. **中间结果访存**：比较`read_MiB`和`write_MiB`，观察融合方案取消卷积中间结果写回和再次读取后，估算GM访问量是否下降。
6. **有效带宽**：`GB/s`由估算搬运量和执行时间共同计算，应结合`kernel_us`及估算读写量分析，不能只根据GB/s的大小判断优化是否有效。
7. **运行稳定性**：观察预热次数、重复次数以及多次运行结果的波动。设备负载、频率和调度状态可能影响计时，应在相同配置和稳定环境下进行比较。

只有在两个实验方案均通过正确性校验后，执行时间、数据搬运量和加速比之间的性能比较才有意义。


### 5.12 Profiling观察

先完成正常构建，再在终端中分别采集基线实验方案和融合优化实验方案：

```bash
cd ~/Source/04.02
bash scripts/profile_msprof_template.sh basic
bash scripts/profile_msprof_template.sh fused
```

分析时重点比较三个Device入口的Kernel执行时间，并观察GM到UB的数据搬入和最终写回。基线方案应出现卷积中间结果的写回与独立ReLU读取；融合方案应只写回最终输出。环形缓冲是否形成有效复用，应结合输入搬入相关指标和Kernel时间线共同判断。


### 5.13 扩展实验

保持`Cout=16`、`K=3`、`padding=1`、`blockDim=8`、预热次数、重复次数和随机种子不变，只改变`Cin`、`H`和`W`，分别观察小规模、中等规模和较大规模输入下的执行时间、估算搬运量和加速比。该扩展实验不改变内部固定宽度块，也不引入宽度块命令行参数。

```bash
cd ~/Source/04.02
mkdir -p results/scale

# 小规模
bash scripts/run_ascend.sh -c -C 4 -O 16 -H 32 -W 32 -K 3 -P 1 -b 8 -V all -w 2 -r 10 -e 1234 \
  | tee results/scale/small.log

# 中等规模
bash scripts/run_ascend.sh -c -C 8 -O 16 -H 64 -W 64 -K 3 -P 1 -b 8 -V all -w 2 -r 10 -e 1234 \
  | tee results/scale/medium.log

# 较大规模
bash scripts/run_ascend.sh -c -C 16 -O 16 -H 96 -W 96 -K 3 -P 1 -b 8 -V all -w 2 -r 10 -e 1234 \
  | tee results/scale/large.log
```

对比时应先确认两种方案均为PASS，再分析规模增大后标量卷积计算占比、输入复用收益和环形缓冲管理开销的变化。扩展实验结果单独保存在`results/scale`，不与本Notebook主实验的一组结果混合。


---
## 6. 实验总结

本实验实现了卷积分类头的基线实验方案与融合优化实验方案。基线方案通过两个Kernel保留卷积中间结果，独立激活Kernel从Global Memory读取并处理；融合方案在一个Kernel内完成卷积、偏置和激活，减少中间结果的Global Memory写回与再次读取。

环形行缓冲区利用相邻输出行卷积窗口重叠的特点，在LocalTensor中保留仍然有效的输入行，只搬入新进入窗口的数据。两种方案保持输出通道划分、固定内部宽度块、权重加载、卷积计算和激活方式一致，使性能差异集中在Kernel融合和输入数据复用上。

实验结果需要先通过最大绝对误差、最大相对误差、错误数量和运行状态验证正确性，再结合执行时间、估算读写量、输入搬入次数、有效带宽和加速比分析两种方案的性能差异。性能结论应限定在具体输入规模、卷积参数和blockDim配置下，并通过重复运行与输入规模扩展实验判断优化收益是否稳定。
